# Phase 3: finetune one model on comparative reasoning

Finetunes a Phase 2 pretrained model on its own language's synthetic
comparative-reasoning dataset, then scores pretrained against finetuned on the
held-out test split and compares their attention.

Run it **once per language** by changing `LANGUAGE` below.

### Before running

* Accelerator: **GPU** (T4 or P100). Settings -> Accelerator.
* **+ Add Data**: the pretrained checkpoint dataset for this language, e.g.
  `lma-phase2-hi-final`.
* The reasoning dataset is generated inside this notebook from the committed
  generator, so nothing else needs attaching - and because generation is seeded,
  it reproduces the corpus in the repository exactly.

### Why this is much cheaper than pretraining

Pretraining consumed ~590M tokens over 12,000 steps. Finetuning sees 16,000 short
examples of about 20 tokens each, so a few epochs is a matter of minutes rather
than hours. The 12-hour session limit is not a constraint here, but the run is
still checkpointed and resumable, because the assignment requires finetuned
checkpoints in the same resume-capable format as pretraining.

## 1. Choose the language

In [ ]:
LANGUAGE = "hi"        # "hi" or "ne" - must match the attached checkpoint

EPOCHS = 3
BATCH_SIZE = 32
LEARNING_RATE = 5e-5
TOTAL_EXAMPLES = 20000   # same default as scripts/generate_reasoning_data.py

assert LANGUAGE in ("hi", "ne")
print(f"finetuning Model {'H (Hindi)' if LANGUAGE == 'hi' else 'L (Nepali)'}")

## 2. The project code

The `lma` package travels inside this notebook rather than being cloned: the
GitHub Classroom repository is private, so an unauthenticated `git clone` from
Kaggle fails. These are the committed sources, written out verbatim.

In [ ]:
import sys
from pathlib import Path

SRC = Path("/kaggle/working/src")
for package in ['lma', 'lma/data', 'lma/eval', 'lma/model', 'lma/reasoning', 'lma/stats', 'lma/tokenizer', 'lma/train']:
    (SRC / package).mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"module directory ready: {SRC}")

In [ ]:
# Empty package markers. These cannot use %%writefile, which errors
# on an empty cell body, so they are created directly.
(SRC / "lma/__init__.py").touch()
(SRC / "lma/tokenizer/__init__.py").touch()
(SRC / "lma/stats/__init__.py").touch()
(SRC / "lma/data/__init__.py").touch()

print("created 4 empty package marker(s)")

In [ ]:
%%writefile /kaggle/working/src/lma/model/__init__.py
"""Decoder-only Transformer implemented from primitive PyTorch layers.

Phase 2 of the project forbids ``nn.Transformer*``, HuggingFace model classes,
and any pre-built attention block, so every tensor operation in the forward pass
is written out explicitly here.
"""

from .config import ModelConfig
from .transformer import (
    CausalSelfAttention,
    DecoderBlock,
    FeedForward,
    TransformerLM,
)

__all__ = [
    "ModelConfig",
    "CausalSelfAttention",
    "FeedForward",
    "DecoderBlock",
    "TransformerLM",
]

In [ ]:
%%writefile /kaggle/working/src/lma/model/config.py
"""Model configuration, with an exact parameter count derived from the shapes.

The assignment fixes a target of roughly 25M trainable parameters per model and
asks for the exact count to be reported, so the arithmetic lives here next to the
hyperparameters rather than being measured only after a model is built. That also
makes it possible to search for a configuration that hits the budget without
allocating anything.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field


@dataclass
class ModelConfig:
    """Hyperparameters for one decoder-only language model.

    The two models in this project share an architecture but not a vocabulary:
    each uses the tokenizer trained on its own corpus in Phase 1, so
    `vocab_size` differs and the embedding and output layers differ in size.

    Attributes:
        vocab_size: Number of tokenizer pieces. From the Phase 1 tokenizer.
        n_layer: Number of stacked decoder blocks.
        n_head: Number of attention heads. Must divide `n_embd`.
        n_embd: Model dimension `d`.
        n_inner: Inner width of the feed-forward network. Defaults to 4x `n_embd`,
            the usual ratio, and the assignment requires it to exceed `n_embd`.
        block_size: Maximum context length in tokens. Learned positional
            embeddings are allocated for exactly this many positions, which is
            the hard limit on sequence length at inference (see `TransformerLM`).
        dropout: Applied to embeddings, attention probabilities, and sublayer
            outputs.
        tie_weights: Share the input embedding matrix with the output projection.
            Saves `vocab_size * n_embd` parameters, which at these sizes is a
            fifth of the whole budget.
        bias: Whether linear layers carry bias terms.
        use_positional_embeddings: Whether position is injected at all. Setting
            this to ``False`` is the assignment's optional ablation: self-attention
            is permutation invariant, so without a positional signal the model
            cannot distinguish orderings of the same tokens. Kept as a flag rather
            than a separate class so the ablated model is the same code path, and
            so a checkpoint records which variant produced it.
    """

    vocab_size: int
    n_layer: int = 12
    n_head: int = 6
    n_embd: int = 384
    n_inner: int | None = None
    block_size: int = 512
    dropout: float = 0.1
    tie_weights: bool = True
    bias: bool = True
    use_positional_embeddings: bool = True

    def __post_init__(self) -> None:
        if self.n_embd % self.n_head != 0:
            raise ValueError(
                f"n_embd ({self.n_embd}) must be divisible by n_head ({self.n_head})"
            )
        if self.n_inner is None:
            self.n_inner = 4 * self.n_embd
        if self.n_inner <= self.n_embd:
            raise ValueError(
                f"n_inner ({self.n_inner}) must exceed n_embd ({self.n_embd}); the "
                "feed-forward network is required to project up before projecting back"
            )

    @property
    def head_dim(self) -> int:
        """Per-head subspace width `d_k = d / h`."""
        return self.n_embd // self.n_head

    def parameter_count(self) -> dict[str, int]:
        """Exact trainable-parameter count, broken down by component.

        Computed from shapes rather than by instantiating the model, so a
        configuration can be checked against the 25M budget cheaply.

        Returns:
            Counts per component plus a ``total`` key.
        """
        d, v, ff = self.n_embd, self.vocab_size, self.n_inner
        b = 1 if self.bias else 0

        token_embedding = v * d
        # The ablation removes the positional table entirely rather than zeroing
        # it, so the parameter count reflects what is actually trained.
        position_embedding = self.block_size * d if self.use_positional_embeddings else 0

        # Per block: four d x d projections (Q, K, V, O) and two feed-forward
        # matrices, plus two LayerNorms (weight and bias each).
        attn = 4 * (d * d + b * d)
        mlp = (d * ff + b * ff) + (ff * d + b * d)
        norms = 2 * (2 * d)
        per_block = attn + mlp + norms
        blocks = self.n_layer * per_block

        final_norm = 2 * d
        # The output projection is free when tied to the input embedding.
        head = 0 if self.tie_weights else v * d

        total = token_embedding + position_embedding + blocks + final_norm + head
        return {
            "token_embedding": token_embedding,
            "position_embedding": position_embedding,
            "blocks": blocks,
            "per_block": per_block,
            "final_norm": final_norm,
            "lm_head": head,
            "total": total,
        }

    @property
    def n_params(self) -> int:
        """Total trainable parameters."""
        return self.parameter_count()["total"]

    def embedding_share(self) -> float:
        """Fraction of the budget spent on embeddings rather than on layers.

        The quantity the Phase 1 vocabulary sweep was chosen against: every piece
        added to the vocabulary is a parameter taken away from the blocks that do
        the modelling.
        """
        counts = self.parameter_count()
        embeddings = counts["token_embedding"] + counts["position_embedding"]
        return embeddings / counts["total"]

    def to_dict(self) -> dict:
        """Serialize for checkpointing and for the run's config record."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "ModelConfig":
        """Rebuild from a checkpoint payload, ignoring unknown keys."""
        fields = {f for f in cls.__dataclass_fields__}
        return cls(**{k: v for k, v in data.items() if k in fields})

In [ ]:
%%writefile /kaggle/working/src/lma/model/transformer.py
"""A decoder-only (GPT-style) Transformer written from primitive layers.

Only ``nn.Linear``, ``nn.Embedding``, ``nn.LayerNorm``, and ``nn.Dropout`` are
used. Attention is computed explicitly - the projection, the head reshape, the
scaled dot product, the causal mask, the softmax, and the output projection are
all written out - because the point of this phase is to be able to explain every
tensor operation in the forward pass.

Shape convention throughout: ``B`` batch, ``T`` time (sequence position), ``d``
model dimension, ``h`` heads, ``dk = d / h`` per-head width.
"""

from __future__ import annotations

import math

import torch
import torch.nn as nn
import torch.nn.functional as F

from .config import ModelConfig


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention.

    Each position attends to itself and to earlier positions only. The mask is
    additive and applied to the scores *before* the softmax, so masked positions
    receive zero probability rather than being renormalized afterwards.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.head_dim = config.head_dim

        # One matrix producing Q, K and V together. Mathematically identical to
        # three separate d x d projections, but a single matmul is faster.
        self.qkv = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)

        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)

        # Upper-triangular mask, held as a buffer so it moves with .to(device)
        # and is saved in no gradient. Bool: True where attention is forbidden.
        mask = torch.triu(
            torch.ones(config.block_size, config.block_size, dtype=torch.bool),
            diagonal=1,
        )
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Apply causal self-attention.

        Args:
            x: Input activations, shape ``(B, T, d)``.
            return_attention: Also return the post-softmax attention
                probabilities, shape ``(B, h, T, T)``. Used by the Phase 2
                attention analysis (heatmaps, entropy, mean distance); left off
                during training because materializing them costs memory.

        Returns:
            ``(output, attention)`` where output has shape ``(B, T, d)`` and
            attention is ``None`` unless requested.
        """
        B, T, d = x.shape

        # Project to queries, keys and values: (B, T, d) -> (B, T, 3d) -> 3 x (B, T, d)
        q, k, v = self.qkv(x).split(self.n_embd, dim=2)

        # Split each into heads and move the head axis in front of time, so that
        # every head is an independent (T, dk) sequence for the matmul below:
        #   (B, T, d) -> (B, T, h, dk) -> (B, h, T, dk)
        q = q.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.head_dim).transpose(1, 2)

        # Scaled dot-product scores: (B, h, T, dk) @ (B, h, dk, T) -> (B, h, T, T).
        #
        # The 1/sqrt(dk) scaling matters. q and k have unit-ish variance per
        # component, so their dot product over dk dimensions has variance
        # proportional to dk. Without the scaling, scores grow with model width,
        # the softmax saturates, and its gradient vanishes - training stalls for
        # exactly the wider models the scaling was meant to enable.
        scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)

        # Causal mask: -inf on future positions, so softmax assigns them exactly
        # zero. Sliced to T because the final batch of a sequence may be shorter.
        scores = scores.masked_fill(self.causal_mask[:T, :T], float("-inf"))

        attn = F.softmax(scores, dim=-1)
        attn = self.attn_dropout(attn)

        # Weighted sum of values: (B, h, T, T) @ (B, h, T, dk) -> (B, h, T, dk)
        y = attn @ v

        # Concatenate heads back together: (B, h, T, dk) -> (B, T, h, dk) -> (B, T, d).
        # contiguous() because transpose leaves a non-contiguous view that view()
        # cannot reinterpret.
        y = y.transpose(1, 2).contiguous().view(B, T, d)

        y = self.resid_dropout(self.proj(y))
        return y, (attn if return_attention else None)


class FeedForward(nn.Module):
    """Position-wise feed-forward network.

    Two linear layers with a GELU between them, applied identically at every
    position. The inner dimension is wider than the model dimension (4x by
    default): this is where most of the block's capacity lives, and where the
    model does per-token computation that attention - which only mixes
    information between positions - cannot do.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.fc = nn.Linear(config.n_embd, config.n_inner, bias=config.bias)
        self.proj = nn.Linear(config.n_inner, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Map ``(B, T, d)`` to ``(B, T, d)`` through the wider inner layer."""
        return self.dropout(self.proj(F.gelu(self.fc(x))))


class DecoderBlock(nn.Module):
    """One pre-norm decoder block: attention, then feed-forward, both residual.

    Pre-norm (normalize *entering* each sublayer, add the raw residual) rather
    than post-norm. With pre-norm the residual path from input to loss carries no
    normalization, so gradients reach early layers undiminished and deep stacks
    train without a warmup-dependent divergence. Post-norm places a LayerNorm on
    that path and needs careful warmup to remain stable at this depth.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = FeedForward(config)

    def forward(
        self, x: torch.Tensor, return_attention: bool = False
    ) -> tuple[torch.Tensor, torch.Tensor | None]:
        """Run both sublayers, each wrapped in a residual connection."""
        attn_out, attn_weights = self.attn(self.ln_1(x), return_attention)
        x = x + attn_out
        x = x + self.mlp(self.ln_2(x))
        return x, attn_weights


class TransformerLM(nn.Module):
    """Decoder-only Transformer language model.

    Learned absolute positional embeddings are used rather than sinusoidal or
    rotary. The reason is that this model is trained and evaluated at a single
    fixed context length, where learned positions are simplest and train fastest;
    the cost is that the model cannot be run on sequences longer than
    `block_size`, because no embedding exists for those positions. Sinusoidal or
    RoPE encodings would extrapolate, at the price of extra machinery this
    project does not need.
    """

    def __init__(self, config: ModelConfig) -> None:
        super().__init__()
        self.config = config

        self.token_embedding = nn.Embedding(config.vocab_size, config.n_embd)
        # The ablation omits the table entirely rather than zeroing it, so the
        # parameters are genuinely absent from the model and from its count.
        self.position_embedding = (
            nn.Embedding(config.block_size, config.n_embd)
            if config.use_positional_embeddings
            else None
        )
        self.embed_dropout = nn.Dropout(config.dropout)

        self.blocks = nn.ModuleList(DecoderBlock(config) for _ in range(config.n_layer))
        self.ln_f = nn.LayerNorm(config.n_embd)
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        if config.tie_weights:
            # One matrix serving as both the input embedding and the output
            # projection. Saves vocab_size x n_embd parameters - about a fifth of
            # this budget - and ties the representation of a token to the vector
            # that predicts it.
            self.lm_head.weight = self.token_embedding.weight

        self.apply(self._init_weights)

        # Scaled init for the residual projections. Each block adds into the
        # residual stream, so with n_layer blocks the variance accumulated at the
        # top grows with depth; scaling these by 1/sqrt(2 * n_layer) keeps
        # activations stable at initialization regardless of depth.
        for name, param in self.named_parameters():
            if name.endswith("proj.weight"):
                nn.init.normal_(
                    param, mean=0.0, std=0.02 / math.sqrt(2 * config.n_layer)
                )

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        """Normal(0, 0.02) for weights, zeros for biases - the GPT-2 scheme."""
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_parameters(self, trainable_only: bool = True) -> int:
        """Count parameters, counting a tied embedding matrix only once."""
        seen: set[int] = set()
        total = 0
        for param in self.parameters():
            if trainable_only and not param.requires_grad:
                continue
            if id(param) in seen:
                continue
            seen.add(id(param))
            total += param.numel()
        return total

    def forward(
        self,
        idx: torch.Tensor,
        targets: torch.Tensor | None = None,
        return_attention: bool = False,
    ) -> dict:
        """Predict next-token logits, and optionally the language-modeling loss.

        Args:
            idx: Token ids, shape ``(B, T)``.
            targets: Next-token ids, shape ``(B, T)``. When given, the causal
                language-modeling loss is returned alongside the logits.
            return_attention: Collect per-layer attention probabilities.

        Returns:
            Dict with ``logits`` ``(B, T, vocab_size)``, ``loss`` (or None), and
            ``attentions`` (list of ``(B, h, T, T)`` tensors, or None).
        """
        B, T = idx.shape
        if T > self.config.block_size:
            raise ValueError(
                f"sequence length {T} exceeds block_size {self.config.block_size}; "
                "learned positional embeddings exist only for block_size positions"
            )

        # Token identity plus position. Addition, not concatenation: the two
        # signals share the same d dimensions, and the model learns to separate
        # them. Without the positional term the model is permutation invariant -
        # attention alone cannot distinguish "dog bites man" from "man bites dog",
        # which is exactly what the ablation is meant to demonstrate.
        x = self.token_embedding(idx)
        if self.position_embedding is not None:
            pos = torch.arange(T, device=idx.device)
            x = x + self.position_embedding(pos)
        x = self.embed_dropout(x)

        attentions = [] if return_attention else None
        for block in self.blocks:
            x, attn = block(x, return_attention)
            if return_attention:
                attentions.append(attn)

        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            # Cross-entropy between the logits at position t and the token at
            # t+1, averaged over every position in the batch. The caller supplies
            # targets already shifted by one.
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.reshape(-1),
                ignore_index=-100,
            )

        return {"logits": logits, "loss": loss, "attentions": attentions}

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int,
        temperature: float = 1.0,
        top_k: int | None = None,
        greedy: bool = False,
        eos_id: int | None = None,
    ) -> torch.Tensor:
        """Autoregressively extend a prompt.

        Args:
            idx: Prompt token ids, shape ``(B, T)``.
            max_new_tokens: How many tokens to append.
            temperature: Divides the logits before softmax. Below 1 sharpens the
                distribution, above 1 flattens it. Ignored when `greedy`.
            top_k: Restrict sampling to the k most likely tokens.
            greedy: Take the argmax instead of sampling.
            eos_id: Stop early once every sequence has produced this token.

        Returns:
            The prompt with the generated continuation appended.
        """
        self.eval()
        finished = torch.zeros(idx.size(0), dtype=torch.bool, device=idx.device)

        for _ in range(max_new_tokens):
            # Crop to the context window: positions beyond block_size have no
            # learned embedding, so the model can only condition on the most
            # recent block_size tokens.
            idx_cond = idx[:, -self.config.block_size :]
            logits = self(idx_cond)["logits"][:, -1, :]

            if greedy:
                next_token = logits.argmax(dim=-1, keepdim=True)
            else:
                logits = logits / max(temperature, 1e-8)
                if top_k is not None:
                    kth = logits.topk(min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
                    logits = logits.masked_fill(logits < kth, float("-inf"))
                next_token = torch.multinomial(F.softmax(logits, dim=-1), num_samples=1)

            if eos_id is not None:
                # Once a sequence has emitted EOS, keep emitting it so the tensor
                # stays rectangular; the caller truncates at the first EOS.
                next_token = torch.where(
                    finished.unsqueeze(1),
                    torch.full_like(next_token, eos_id),
                    next_token,
                )
                finished |= next_token.squeeze(1) == eos_id

            idx = torch.cat([idx, next_token], dim=1)
            if eos_id is not None and bool(finished.all()):
                break

        return idx

In [ ]:
%%writefile /kaggle/working/src/lma/train/__init__.py
"""Pretraining: batching, optimization, and resumable checkpoints."""

from .data import TokenBinaryDataset
from .trainer import TrainConfig, Trainer, TrainState, build_optimizer, cosine_lr

__all__ = [
    "TokenBinaryDataset",
    "TrainConfig",
    "Trainer",
    "TrainState",
    "build_optimizer",
    "cosine_lr",
]

In [ ]:
%%writefile /kaggle/working/src/lma/train/data.py
"""Batching for causal language-model pretraining.

The corpus is one flat ``uint16`` array of token ids on disk, produced by the
tokenization step. Batches are random windows into that array, memory-mapped so
the file is never loaded whole - a 1.2 GB corpus streams from page cache while
resident memory stays near zero.
"""

from __future__ import annotations

from pathlib import Path

import numpy as np
import torch


class TokenBinaryDataset:
    """Random fixed-length windows over a flat array of token ids.

    Two details that matter for correctness:

    * **Targets are inputs shifted by one.** A window of ``block_size + 1``
      tokens yields ``x = w[:-1]`` and ``y = w[1:]``, so position ``t`` predicts
      the token at ``t + 1``. Sampling ``block_size + 1`` rather than
      ``block_size`` avoids the off-by-one that silently trains the model to
      predict its own input.
    * **Windows may cross document boundaries.** Documents were concatenated with
      an EOS token between them, so the model sees EOS as an ordinary token and
      learns that text ends there. This is standard for pretraining and means no
      document structure has to be tracked at batch time.
    """

    def __init__(self, path: str | Path, block_size: int, seed: int = 0) -> None:
        """Open a token binary for sampling.

        Args:
            path: ``.bin`` file of ``uint16`` token ids.
            block_size: Context length. Windows are ``block_size + 1`` long.
            seed: Seeds this dataset's own generator, so batch order is
                reproducible and independent of global torch/numpy state.
        """
        self.path = Path(path)
        if not self.path.exists():
            raise FileNotFoundError(f"token binary not found: {self.path}")

        self.block_size = block_size
        # mode="r" keeps the file read-only and lets several workers - or both
        # languages' loaders - share the same page cache.
        self.tokens = np.memmap(self.path, dtype=np.uint16, mode="r")
        self.rng = np.random.default_rng(seed)

        if len(self.tokens) <= block_size + 1:
            raise ValueError(
                f"{self.path.name} holds {len(self.tokens):,} tokens, too few for "
                f"block_size {block_size}"
            )

    def __len__(self) -> int:
        """Number of distinct starting positions."""
        return len(self.tokens) - self.block_size - 1

    def batch(self, batch_size: int, device: str = "cpu") -> tuple[torch.Tensor, torch.Tensor]:
        """Draw one batch of random windows.

        Returns:
            ``(x, y)``, both ``(batch_size, block_size)`` int64 tensors, with
            ``y`` shifted one position ahead of ``x``.
        """
        starts = self.rng.integers(0, len(self), size=batch_size)

        # Build in numpy, convert once: per-sample torch indexing is markedly
        # slower than one contiguous copy.
        window = np.stack(
            [self.tokens[s : s + self.block_size + 1].astype(np.int64) for s in starts]
        )
        batch = torch.from_numpy(window)

        x = batch[:, :-1].contiguous()
        y = batch[:, 1:].contiguous()

        if device != "cpu":
            # non_blocking with pinned memory overlaps the host-to-device copy
            # with compute on the previous batch.
            x = x.pin_memory().to(device, non_blocking=True)
            y = y.pin_memory().to(device, non_blocking=True)
        return x, y

    def sequential_batches(
        self, batch_size: int, limit: int | None = None, device: str = "cpu"
    ):
        """Iterate the corpus in order, for deterministic evaluation.

        Validation loss should not depend on a random draw, so held-out
        evaluation walks contiguous non-overlapping windows from the start of the
        file instead of sampling.

        Args:
            batch_size: Windows per batch.
            limit: Stop after this many batches. ``None`` walks the whole split.
            device: Where to place the tensors.

        Yields:
            ``(x, y)`` pairs shaped as in :meth:`batch`.
        """
        stride = self.block_size
        position = 0
        produced = 0

        while position + (batch_size * stride) + 1 <= len(self.tokens):
            if limit is not None and produced >= limit:
                return

            starts = [position + i * stride for i in range(batch_size)]
            window = np.stack(
                [self.tokens[s : s + self.block_size + 1].astype(np.int64) for s in starts]
            )
            batch = torch.from_numpy(window)

            x = batch[:, :-1].contiguous()
            y = batch[:, 1:].contiguous()
            if device != "cpu":
                x = x.pin_memory().to(device, non_blocking=True)
                y = y.pin_memory().to(device, non_blocking=True)

            yield x, y
            position += batch_size * stride
            produced += 1

    def get_rng_state(self) -> dict:
        """Sampler RNG state, for checkpointing.

        The dataset draws window offsets from its own generator rather than the
        global one, so resuming a run has to restore this too - otherwise a
        resumed run sees a different batch sequence than an uninterrupted one and
        the two diverge.
        """
        return self.rng.bit_generator.state

    def set_rng_state(self, state: dict) -> None:
        """Restore a sampler RNG state saved by :meth:`get_rng_state`."""
        self.rng.bit_generator.state = state

    def stats(self) -> dict:
        """Summary for the run log."""
        return {
            "path": str(self.path),
            "tokens": int(len(self.tokens)),
            "bytes": self.path.stat().st_size,
            "windows": len(self),
            "block_size": self.block_size,
        }

In [ ]:
%%writefile /kaggle/working/src/lma/train/trainer.py
"""Pretraining loop for the Phase 2 language models.

Built around one constraint that shapes every design choice here: Kaggle kills a
session after 12 hours, and a full run needs more than that. So training is
resumable at any step - checkpoints carry model weights, optimizer state,
scheduler position, step counter, RNG state, and the config - and a fresh session
picks up exactly where the last one stopped rather than restarting.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import torch
import torch.nn as nn

from ..model import ModelConfig, TransformerLM
from .data import TokenBinaryDataset


@dataclass
class TrainConfig:
    """Optimization hyperparameters for one pretraining run."""

    batch_size: int = 24
    gradient_accumulation: int = 4
    learning_rate: float = 6e-4
    min_learning_rate: float = 6e-5
    warmup_steps: int = 500
    max_steps: int = 12_000
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    eval_interval: int = 500
    eval_batches: int = 100
    checkpoint_interval: int = 1_000
    log_interval: int = 20
    seed: int = 20260812
    amp: bool = True
    compile: bool = False

    @property
    def tokens_per_step(self) -> int:
        """Tokens consumed per optimizer step, ignoring block_size."""
        return self.batch_size * self.gradient_accumulation

    def to_dict(self) -> dict:
        """Serialize for checkpointing and for the run's config record."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "TrainConfig":
        """Rebuild from a checkpoint payload, ignoring unknown keys.

        Unknown keys are dropped rather than raising, so a checkpoint written by
        an earlier version - before a field existed, or after one was removed -
        still loads instead of stranding a trained model.
        """
        fields = set(cls.__dataclass_fields__)
        return cls(**{k: v for k, v in data.items() if k in fields})


def cosine_lr(step: int, cfg: TrainConfig) -> float:
    """Learning rate at `step`: linear warmup, then cosine decay.

    Warmup exists because Adam's second-moment estimate is unreliable in the
    first few hundred steps; starting at full learning rate there produces the
    large early updates that destabilize deep pre-norm stacks. Cosine decay
    afterwards spends most of the budget at a high rate and anneals smoothly,
    which reliably beats a constant rate at a fixed step count.
    """
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / max(cfg.warmup_steps, 1)

    progress = (step - cfg.warmup_steps) / max(cfg.max_steps - cfg.warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.min_learning_rate + coeff * (cfg.learning_rate - cfg.min_learning_rate)


def build_optimizer(model: nn.Module, cfg: TrainConfig) -> torch.optim.AdamW:
    """AdamW with weight decay applied only where it belongs.

    Decaying biases, LayerNorm gains, and embeddings hurts: those parameters set
    scale and offset rather than mixing features, and shrinking them toward zero
    is a bias with no regularizing benefit. Only matrices that participate in
    matrix multiplications - the 2D weights - are decayed.
    """
    decay, no_decay = [], []
    seen: set[int] = set()

    for module in model.modules():
        for name, param in module.named_parameters(recurse=False):
            if not param.requires_grad or id(param) in seen:
                continue
            seen.add(id(param))
            if param.dim() >= 2 and not isinstance(module, nn.Embedding):
                decay.append(param)
            else:
                no_decay.append(param)

    groups = [
        {"params": decay, "weight_decay": cfg.weight_decay},
        {"params": no_decay, "weight_decay": 0.0},
    ]
    # fused AdamW is markedly faster on CUDA and a no-op elsewhere.
    fused = torch.cuda.is_available()
    return torch.optim.AdamW(
        groups, lr=cfg.learning_rate, betas=(cfg.beta1, cfg.beta2), fused=fused
    )


def _as_byte_tensor(state) -> torch.Tensor:
    """Coerce a saved RNG state into the CPU uint8 tensor torch requires.

    ``torch.set_rng_state`` and ``torch.cuda.set_rng_state_all`` accept only a
    ByteTensor on CPU, and raise ``TypeError: RNG state must be a
    torch.ByteTensor`` otherwise. A state that has been through
    ``torch.save``/``torch.load`` - especially across torch versions or between
    hosts - does not always come back in that exact form.
    """
    if not isinstance(state, torch.Tensor):
        state = torch.tensor(state, dtype=torch.uint8)
    return state.cpu().to(torch.uint8)


@dataclass
class TrainState:
    """Everything needed to resume a run mid-flight."""

    step: int = 0
    tokens_seen: int = 0
    best_val_loss: float = float("inf")
    history: list[dict] = field(default_factory=list)
    wall_seconds: float = 0.0


class Trainer:
    """Pretrain one language model, resumably.

    Args:
        model: The model to train.
        train_data: Sampler over the training binary.
        val_data: Sampler over the validation binary.
        cfg: Optimization hyperparameters.
        out_dir: Where checkpoints and the run log are written.
        device: ``"cuda"`` or ``"cpu"``.
    """

    def __init__(
        self,
        model: TransformerLM,
        train_data: TokenBinaryDataset,
        val_data: TokenBinaryDataset,
        cfg: TrainConfig,
        out_dir: str | Path,
        device: str | None = None,
    ) -> None:
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_data = train_data
        self.val_data = val_data
        self.cfg = cfg
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

        self.optimizer = build_optimizer(self.model, cfg)
        self.state = TrainState()

        # Mixed precision: bfloat16 where supported (no loss scaling needed,
        # same exponent range as fp32), otherwise fp16 with a gradient scaler to
        # keep small gradients from flushing to zero.
        self.amp_enabled = cfg.amp and self.device == "cuda"
        self.amp_dtype = torch.float32
        if self.amp_enabled:
            supported = torch.cuda.is_bf16_supported()
            self.amp_dtype = torch.bfloat16 if supported else torch.float16
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=self.amp_enabled and self.amp_dtype is torch.float16
        )

        if cfg.compile and hasattr(torch, "compile"):
            self.model = torch.compile(self.model)

        torch.manual_seed(cfg.seed)

    # -- evaluation ---------------------------------------------------------

    @torch.no_grad()
    def evaluate(self, batches: int | None = None) -> dict:
        """Mean cross-entropy over held-out data, and the perplexity implied.

        Walks the validation split in order rather than sampling, so the number
        is comparable across steps and across runs.
        """
        self.model.eval()
        n_batches = batches if batches is not None else self.cfg.eval_batches

        losses = []
        for x, y in self.val_data.sequential_batches(
            self.cfg.batch_size, limit=n_batches, device=self.device
        ):
            with torch.autocast(
                device_type=self.device, dtype=self.amp_dtype, enabled=self.amp_enabled
            ):
                loss = self.model(x, targets=y)["loss"]
            losses.append(loss.float().item())

        self.model.train()
        mean_loss = sum(losses) / max(len(losses), 1)
        return {
            "val_loss": mean_loss,
            # Perplexity is exp of the mean cross-entropy, per the assignment.
            "val_perplexity": math.exp(min(mean_loss, 20)),
            "batches": len(losses),
        }

    # -- checkpointing ------------------------------------------------------

    def save_checkpoint(self, name: str = "latest") -> Path:
        """Write a checkpoint that a fresh session can resume from.

        Carries model weights, optimizer state, the step counter, and the config,
        as the assignment requires - plus RNG state, so a resumed run draws the
        same batch sequence it would have drawn had it never stopped. The
        learning-rate schedule is a pure function of the step, so storing the step
        stores the scheduler position.
        """
        model = getattr(self.model, "_orig_mod", self.model)  # unwrap torch.compile
        payload = {
            "model": model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
            "step": self.state.step,
            "tokens_seen": self.state.tokens_seen,
            "best_val_loss": self.state.best_val_loss,
            "history": self.state.history,
            "wall_seconds": self.state.wall_seconds,
            "model_config": model.config.to_dict(),
            "train_config": self.cfg.to_dict(),
            # The sampler's own generator, without which a resumed run draws a
            # different batch sequence and diverges from an uninterrupted one.
            "train_sampler_rng": self.train_data.get_rng_state(),
            "torch_rng": torch.get_rng_state(),
            "cuda_rng": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
        }
        path = self.out_dir / f"checkpoint_{name}.pt"
        # Write to a temporary file first: a session killed mid-write would
        # otherwise leave a truncated checkpoint and lose the whole run.
        tmp = path.with_suffix(".pt.tmp")
        torch.save(payload, tmp)
        tmp.replace(path)
        return path

    def load_checkpoint(self, path: str | Path) -> None:
        """Restore a run from a checkpoint written by :meth:`save_checkpoint`."""
        payload = torch.load(path, map_location=self.device, weights_only=False)
        model = getattr(self.model, "_orig_mod", self.model)
        model.load_state_dict(payload["model"])
        self.optimizer.load_state_dict(payload["optimizer"])
        if payload.get("scaler"):
            self.scaler.load_state_dict(payload["scaler"])

        self.state = TrainState(
            step=payload["step"],
            tokens_seen=payload["tokens_seen"],
            best_val_loss=payload["best_val_loss"],
            history=payload.get("history", []),
            wall_seconds=payload.get("wall_seconds", 0.0),
        )

        # Restoring RNG state makes a resumed run follow the same trajectory as an
        # uninterrupted one. It is a fidelity improvement, not a correctness
        # requirement: the model, optimizer, and step above are what actually
        # continue the run. So every restore below is best-effort - a checkpoint
        # written under a different torch version or GPU count can carry states
        # this build will not accept, and refusing to resume over that would
        # throw away a valid checkpoint for no benefit.
        if payload.get("train_sampler_rng") is not None:
            try:
                self.train_data.set_rng_state(payload["train_sampler_rng"])
            except (ValueError, KeyError, TypeError) as error:
                print(f"  note: sampler RNG state not restored ({error})")

        if payload.get("torch_rng") is not None:
            try:
                torch.set_rng_state(_as_byte_tensor(payload["torch_rng"]))
            except (RuntimeError, TypeError) as error:
                print(f"  note: CPU RNG state not restored ({error})")

        if payload.get("cuda_rng") and torch.cuda.is_available():
            try:
                states = [_as_byte_tensor(state) for state in payload["cuda_rng"]]
                # set_rng_state_all expects exactly one state per visible device;
                # a checkpoint from a host with a different GPU count would
                # otherwise raise deep inside the CUDA lazy-init callback.
                if len(states) == torch.cuda.device_count():
                    torch.cuda.set_rng_state_all(states)
                else:
                    print(
                        f"  note: CUDA RNG state not restored "
                        f"({len(states)} saved, {torch.cuda.device_count()} device(s) here)"
                    )
            except (RuntimeError, TypeError) as error:
                print(f"  note: CUDA RNG state not restored ({error})")

    def find_resume_point(self) -> Path | None:
        """Latest checkpoint in the output directory, if any."""
        latest = self.out_dir / "checkpoint_latest.pt"
        return latest if latest.exists() else None

    # -- training -----------------------------------------------------------

    def train(self, max_seconds: float | None = None, resume: bool = True) -> TrainState:
        """Run the training loop.

        Args:
            max_seconds: Stop cleanly after this much wall time, saving a
                checkpoint first. Set it below the platform's session limit so
                the run ends on its own terms rather than being killed mid-step.
            resume: Continue from ``checkpoint_latest.pt`` when one exists.

        Returns:
            The final training state.
        """
        if resume:
            found = self.find_resume_point()
            if found:
                self.load_checkpoint(found)
                print(f"resumed from {found.name} at step {self.state.step:,}")

        cfg = self.cfg
        block = self.train_data.block_size
        self.model.train()
        started = time.time()
        window_start = time.time()
        window_loss = 0.0
        window_steps = 0

        while self.state.step < cfg.max_steps:
            lr = cosine_lr(self.state.step, cfg)
            for group in self.optimizer.param_groups:
                group["lr"] = lr

            # Gradient accumulation: several micro-batches summed into one
            # update, so the effective batch can exceed what fits in memory.
            self.optimizer.zero_grad(set_to_none=True)
            step_loss = 0.0

            for _ in range(cfg.gradient_accumulation):
                x, y = self.train_data.batch(cfg.batch_size, device=self.device)
                with torch.autocast(
                    device_type=self.device, dtype=self.amp_dtype, enabled=self.amp_enabled
                ):
                    loss = self.model(x, targets=y)["loss"]
                    # Divide so the accumulated gradient is the mean over
                    # micro-batches, not their sum.
                    loss = loss / cfg.gradient_accumulation

                self.scaler.scale(loss).backward()
                step_loss += loss.float().item()

            if cfg.grad_clip > 0:
                # Unscale before clipping, or the threshold would apply to
                # scaled gradients and mean nothing.
                self.scaler.unscale_(self.optimizer)
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), cfg.grad_clip)

            self.scaler.step(self.optimizer)
            self.scaler.update()

            self.state.step += 1
            self.state.tokens_seen += cfg.tokens_per_step * block
            window_loss += step_loss
            window_steps += 1

            if self.state.step % cfg.log_interval == 0:
                elapsed = time.time() - window_start
                tps = (cfg.tokens_per_step * block * window_steps) / max(elapsed, 1e-9)
                print(
                    f"step {self.state.step:>6,}/{cfg.max_steps:,}  "
                    f"loss {window_loss / window_steps:.4f}  "
                    f"lr {lr:.2e}  {tps / 1e3:.1f}k tok/s  "
                    f"seen {self.state.tokens_seen / 1e6:.0f}M",
                    flush=True,
                )
                window_loss = 0.0
                window_steps = 0
                window_start = time.time()

            if self.state.step % cfg.eval_interval == 0:
                metrics = self.evaluate()
                metrics.update(step=self.state.step, lr=lr, tokens_seen=self.state.tokens_seen)
                self.state.history.append(metrics)
                print(
                    f"  eval step {self.state.step:,}: "
                    f"val_loss {metrics['val_loss']:.4f}  "
                    f"ppl {metrics['val_perplexity']:.2f}",
                    flush=True,
                )
                if metrics["val_loss"] < self.state.best_val_loss:
                    self.state.best_val_loss = metrics["val_loss"]
                    self.save_checkpoint("best")
                self._write_history()

            if self.state.step % cfg.checkpoint_interval == 0:
                self.save_checkpoint("latest")

            if max_seconds is not None and (time.time() - started) > max_seconds:
                self.state.wall_seconds += time.time() - started
                self.save_checkpoint("latest")
                self._write_history()
                print(
                    f"\nstopped at step {self.state.step:,} after "
                    f"{(time.time() - started) / 60:.0f} min (time budget reached); "
                    "rerun to continue from this checkpoint"
                )
                return self.state

        self.state.wall_seconds += time.time() - started
        self.save_checkpoint("latest")
        final = self.evaluate()
        final.update(step=self.state.step, tokens_seen=self.state.tokens_seen, final=True)
        self.state.history.append(final)
        self._write_history()
        print(
            f"\ntraining complete at step {self.state.step:,}: "
            f"val_loss {final['val_loss']:.4f}  ppl {final['val_perplexity']:.2f}"
        )
        return self.state

    def _write_history(self) -> None:
        """Persist the metric history for the report's loss curves."""
        path = self.out_dir / "training_history.json"
        path.write_text(
            json.dumps(
                {
                    "step": self.state.step,
                    "tokens_seen": self.state.tokens_seen,
                    "best_val_loss": self.state.best_val_loss,
                    "wall_seconds": round(self.state.wall_seconds, 1),
                    "train_config": self.cfg.to_dict(),
                    "history": self.state.history,
                },
                indent=1,
            )
        )

In [ ]:
%%writefile /kaggle/working/src/lma/tokenizer/train_tokenizer.py
"""Training subword tokenizers from scratch with SentencePiece.

Phase 1 forbids pretrained tokenizers, so every vocabulary here is fitted on the
language's own training split only. The two languages get entirely separate
tokenizers and vocabularies - nothing is shared, not even the special-token ids
by coincidence of a shared file.

Model choice
------------
Both BPE and Unigram are supported behind one interface so the choice can be
made empirically (see :mod:`lma.stats.tokenizer_stats`) rather than by assertion.
Unigram is the usual better fit for agglutinative Indic morphology because its
pruning objective yields more linguistically coherent pieces, while BPE tends to
produce shorter sequences for a given vocabulary size. The selection script
reports fertility and unknown-token rate for both so the report can justify the
final pick.

Character coverage
------------------
Defaults to 0.9995 rather than 1.0. Bengali-script corpora contain a long tail
of stray Latin, CJK, and symbol characters; at full coverage each becomes its own
vocabulary entry, wasting slots. The remainder falls back to byte pieces, so no
input is ever unrepresentable.
"""

from __future__ import annotations

import os
import shutil
import tempfile
from dataclasses import dataclass, asdict, field
from pathlib import Path
from typing import Iterable, Iterator

import sentencepiece as spm

#: Special tokens and their fixed ids, identical in structure per language but
#: fitted on separate corpora. BOS/EOS are needed for Phase 2 sequence framing;
#: PAD keeps batches rectangular.
PAD_TOKEN, UNK_TOKEN, BOS_TOKEN, EOS_TOKEN = "<pad>", "<unk>", "<s>", "</s>"
PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3


@dataclass
class TokenizerConfig:
    """Hyperparameters for one tokenizer training run.

    Attributes:
        language: ISO code, used in output filenames.
        model_type: ``"unigram"`` or ``"bpe"``.
        vocab_size: Target vocabulary size, including special tokens.
        character_coverage: Fraction of training characters the vocabulary must
            cover before falling back to byte pieces.
        input_sentence_size: Cap on sentences sampled for training; SentencePiece
            holds these in memory, so this bounds RAM on a large corpus.
        shuffle_input_sentence: Sample uniformly across the corpus rather than
            taking a prefix, so one large source cannot dominate the vocabulary.
        max_sentence_length: Bytes; longer lines are split by SentencePiece.
        byte_fallback: Encode out-of-vocabulary characters as UTF-8 byte pieces so
            the unknown rate on held-out text is effectively zero.
        split_digits: Give each digit its own piece, which generalizes numbers.
        allow_whitespace_only_pieces: Keep pieces made only of whitespace.
        normalization_rule_name: SentencePiece's own normalizer. Set to
            ``identity`` because :mod:`lma.data.normalize` already normalized the
            corpus, and a second, script-unaware normalization pass risks folding
            the Assamese-specific letters.
        num_threads: Training parallelism.
        seed: Reproducibility seed.
        user_defined_symbols: Symbols forced into the vocabulary intact.
    """

    language: str
    model_type: str = "unigram"
    vocab_size: int = 32_000
    character_coverage: float = 0.9995
    input_sentence_size: int = 4_000_000
    shuffle_input_sentence: bool = True
    max_sentence_length: int = 16_384
    byte_fallback: bool = True
    split_digits: bool = True
    allow_whitespace_only_pieces: bool = True
    normalization_rule_name: str = "identity"
    num_threads: int = 8
    seed: int = 20260812
    user_defined_symbols: list[str] = field(default_factory=lambda: ["।", "॥"])

    def __post_init__(self) -> None:
        """Validate the model type."""
        if self.model_type not in {"unigram", "bpe"}:
            raise ValueError(f"model_type must be 'unigram' or 'bpe', got {self.model_type!r}")


class TokenizerTrainer:
    """Trains and loads a SentencePiece tokenizer for one language."""

    def __init__(self, config: TokenizerConfig) -> None:
        """Store the training configuration."""
        self.config = config

    def _model_prefix(self, output_dir: Path) -> Path:
        """Return the output path prefix for the trained model files."""
        cfg = self.config
        return Path(output_dir) / f"{cfg.language}_{cfg.model_type}_{cfg.vocab_size}"

    def train(self, corpus_files: Iterable[Path], output_dir: Path) -> dict:
        """Fit a tokenizer on the given corpus files.

        Args:
            corpus_files: Plain-text files, one sentence or document per line.
                These must come from the *training* split only.
            output_dir: Directory to write ``.model`` and ``.vocab`` into.

        Returns:
            A manifest dict with the resolved config and output paths.
        """
        cfg = self.config
        output_dir = Path(output_dir)
        output_dir.mkdir(parents=True, exist_ok=True)
        prefix = self._model_prefix(output_dir)
        files = [str(Path(p)) for p in corpus_files]
        if not files:
            raise ValueError("No corpus files supplied for tokenizer training")

        # SentencePiece takes the RNG seed from the environment rather than from
        # TrainerSpec (there is no `random_seed` trainer field), and it is read at
        # trainer construction, so it must be set before the call below. This is
        # what makes sentence sampling - and therefore the vocabulary - reproducible.
        os.environ["PYTHONHASHSEED"] = str(cfg.seed)
        spm.set_random_generator_seed(cfg.seed)

        spm.SentencePieceTrainer.train(
            input=",".join(files),
            model_prefix=str(prefix),
            model_type=cfg.model_type,
            vocab_size=cfg.vocab_size,
            character_coverage=cfg.character_coverage,
            input_sentence_size=cfg.input_sentence_size,
            shuffle_input_sentence=cfg.shuffle_input_sentence,
            max_sentence_length=cfg.max_sentence_length,
            byte_fallback=cfg.byte_fallback,
            split_digits=cfg.split_digits,
            allow_whitespace_only_pieces=cfg.allow_whitespace_only_pieces,
            normalization_rule_name=cfg.normalization_rule_name,
            num_threads=cfg.num_threads,
            train_extremely_large_corpus=False,
            pad_id=PAD_ID, pad_piece=PAD_TOKEN,
            unk_id=UNK_ID, unk_piece=UNK_TOKEN,
            bos_id=BOS_ID, bos_piece=BOS_TOKEN,
            eos_id=EOS_ID, eos_piece=EOS_TOKEN,
            user_defined_symbols=cfg.user_defined_symbols,
            minloglevel=1,
        )

        return {
            "config": asdict(cfg),
            "model_file": str(prefix.with_suffix(".model")),
            "vocab_file": str(prefix.with_suffix(".vocab")),
            "corpus_files": files,
        }

    @staticmethod
    def load(model_file: Path) -> spm.SentencePieceProcessor:
        """Load a trained tokenizer from its ``.model`` file."""
        processor = spm.SentencePieceProcessor()
        processor.load(str(model_file))
        return processor


def write_training_text(
    texts: Iterable[str],
    output_file: Path,
    max_lines: int | None = None,
    max_line_bytes: int = 12_000,
) -> int:
    """Flatten documents into the one-line-per-sentence file SentencePiece wants.

    Documents are split on newlines and long lines are chunked, because
    SentencePiece silently drops lines over ``max_sentence_length`` - which would
    quietly discard the longest documents in the corpus.

    The chunk limit is measured in **UTF-8 bytes, not characters**, because
    SentencePiece's limit is a byte limit. Bengali-script characters occupy three
    bytes each, so a character-based limit would be roughly 3x larger than
    intended and would still lose long lines. The default of 12,000 bytes leaves
    headroom under the trainer's 16,384-byte default.

    Args:
        texts: Document texts.
        output_file: Destination text file.
        max_lines: Stop after writing this many lines.
        max_line_bytes: Hard-wrap lines longer than this at a space boundary.

    Returns:
        Number of lines written.
    """
    output_file = Path(output_file)
    output_file.parent.mkdir(parents=True, exist_ok=True)
    written = 0

    with output_file.open("w", encoding="utf-8") as handle:
        for text in texts:
            for raw_line in text.split("\n"):
                line = " ".join(raw_line.split())
                if not line:
                    continue
                for chunk in _chunk_line(line, max_line_bytes):
                    handle.write(chunk + "\n")
                    written += 1
                    if max_lines is not None and written >= max_lines:
                        return written
    return written


def _chunk_line(line: str, byte_limit: int) -> Iterator[str]:
    """Split an over-long line at space boundaries into <= `byte_limit` byte chunks.

    A single word longer than the limit is emitted on its own: breaking mid-word
    would fabricate tokens that do not occur in the language, which is worse for a
    vocabulary than one over-long line that SentencePiece may skip.
    """
    if len(line.encode("utf-8")) <= byte_limit:
        yield line
        return

    current: list[str] = []
    length = 0
    for word in line.split(" "):
        word_bytes = len(word.encode("utf-8")) + 1  # +1 for the joining space
        if length + word_bytes > byte_limit and current:
            yield " ".join(current)
            current, length = [], 0
        current.append(word)
        length += word_bytes
    if current:
        yield " ".join(current)


def train_from_jsonl(
    jsonl_files: Iterable[Path],
    config: TokenizerConfig,
    output_dir: Path,
    max_lines: int | None = None,
) -> dict:
    """Convenience wrapper: JSONL training split in, trained tokenizer out.

    Args:
        jsonl_files: Training-split JSONL files.
        config: Tokenizer hyperparameters.
        output_dir: Where to write the model and vocabulary.
        max_lines: Cap on training lines extracted from the corpus.

    Returns:
        The training manifest, with the line count added.
    """
    from ..data.splits import iter_texts

    tmp_dir = Path(tempfile.mkdtemp(prefix="spm_corpus_"))
    try:
        corpus_txt = tmp_dir / f"{config.language}_spm_input.txt"
        lines = write_training_text(iter_texts(jsonl_files), corpus_txt, max_lines=max_lines)
        if lines == 0:
            raise ValueError("Tokenizer training corpus is empty")
        manifest = TokenizerTrainer(config).train([corpus_txt], output_dir)
        manifest["training_lines"] = lines
        manifest["corpus_files"] = [str(Path(p)) for p in jsonl_files]
        return manifest
    finally:
        shutil.rmtree(tmp_dir, ignore_errors=True)

In [ ]:
%%writefile /kaggle/working/src/lma/eval/__init__.py
"""Phase 2 evaluation: language-modeling metrics, generation quality, attention."""

from .attention import (
    attention_statistics,
    classify_heads,
    collect_attention,
    plot_attention_heatmaps,
    plot_head_summary,
)
from .generation import (
    chrf_score,
    corpus_bleu,
    diversity_metrics,
    evaluate_generations,
    rouge_l,
)
from .lm_metrics import compare_models, evaluate_language_model

__all__ = [
    "evaluate_language_model",
    "compare_models",
    "corpus_bleu",
    "chrf_score",
    "rouge_l",
    "diversity_metrics",
    "evaluate_generations",
    "attention_statistics",
    "classify_heads",
    "collect_attention",
    "plot_attention_heatmaps",
    "plot_head_summary",
]

In [ ]:
%%writefile /kaggle/working/src/lma/eval/attention.py
"""What the attention heads learned: entropy, distance, and heatmaps.

Two scalar summaries per head, both computed only over positions a causal model
may actually attend to:

* **Entropy** measures how spread a head's attention is. A head attending to one
  position has entropy 0; one spreading uniformly over its context has the
  maximum for that width. Because a causal head at position ``t`` can only see
  ``t + 1`` positions, raw entropy grows along the sequence for purely structural
  reasons, so the normalized form - entropy divided by ``log(t + 1)`` - is
  reported alongside it and is the figure to compare across heads.

* **Mean attention distance** is how far back a head looks on average, weighted
  by attention mass. Small values mark local heads (previous-token heads, which
  most models learn early); large values mark heads gathering long-range context.
"""

from __future__ import annotations

import math
from pathlib import Path

import torch


@torch.no_grad()
def attention_statistics(attentions: list[torch.Tensor]) -> dict:
    """Per-head entropy and mean attention distance.

    Args:
        attentions: One ``(B, h, T, T)`` tensor per layer, as returned by
            ``TransformerLM(..., return_attention=True)``.

    Returns:
        Dict with ``entropy``, ``normalized_entropy``, and ``mean_distance``,
        each a ``(n_layer, n_head)`` nested list, plus per-layer means.
    """
    entropy_per_layer = []
    normalized_per_layer = []
    distance_per_layer = []

    for layer_attention in attentions:
        B, H, T, _ = layer_attention.shape
        weights = layer_attention.float()

        # Entropy per query position: -sum p log p. The clamp keeps log(0)
        # finite; masked positions carry exactly zero probability and so
        # contribute nothing to the sum either way.
        logs = torch.log(weights.clamp_min(1e-12))
        entropy = -(weights * logs).sum(dim=-1)          # (B, H, T)

        # Position t attends over t+1 positions, so the maximum possible entropy
        # there is log(t+1). Dividing by it makes heads comparable across
        # positions and sequence lengths.
        positions = torch.arange(T, device=weights.device, dtype=torch.float32)
        max_entropy = torch.log(positions + 1).clamp_min(1e-12)
        normalized = entropy / max_entropy

        # Distance from each query to each key, in positions. Only the causal
        # (lower-triangular) part carries mass, so the weighted sum is the mean
        # look-back distance.
        query_index = positions.view(1, 1, T, 1)
        key_index = positions.view(1, 1, 1, T)
        distance = (query_index - key_index).clamp_min(0)
        mean_distance = (weights * distance).sum(dim=-1)  # (B, H, T)

        # Position 0 can only attend to itself, so its entropy and distance are
        # trivially zero and would drag every head's mean down equally. Excluded.
        entropy_per_layer.append(entropy[:, :, 1:].mean(dim=(0, 2)).tolist())
        normalized_per_layer.append(normalized[:, :, 1:].mean(dim=(0, 2)).tolist())
        distance_per_layer.append(mean_distance[:, :, 1:].mean(dim=(0, 2)).tolist())

    return {
        "entropy": entropy_per_layer,
        "normalized_entropy": normalized_per_layer,
        "mean_distance": distance_per_layer,
        "layer_mean_entropy": [sum(e) / len(e) for e in entropy_per_layer],
        "layer_mean_distance": [sum(d) / len(d) for d in distance_per_layer],
        "n_layer": len(attentions),
        "n_head": len(entropy_per_layer[0]) if entropy_per_layer else 0,
    }


def classify_heads(stats: dict, local_threshold: float = 5.0) -> dict:
    """Sort heads into local, diffuse, and long-range groups.

    A crude but useful summary for the report's discussion: heads whose mean
    attention distance is small are doing positional work (typically attending to
    the previous token or two), while heads with high normalized entropy are
    spreading mass broadly rather than selecting particular positions.

    Args:
        stats: Output of :func:`attention_statistics`.
        local_threshold: Mean distance below which a head counts as local.
    """
    local, long_range, diffuse = [], [], []

    for layer, (distances, normalized) in enumerate(
        zip(stats["mean_distance"], stats["normalized_entropy"])
    ):
        for head, (distance, entropy) in enumerate(zip(distances, normalized)):
            record = {
                "layer": layer,
                "head": head,
                "mean_distance": distance,
                "normalized_entropy": entropy,
            }
            if distance < local_threshold:
                local.append(record)
            elif entropy > 0.8:
                diffuse.append(record)
            else:
                long_range.append(record)

    return {
        "local": local,
        "long_range": long_range,
        "diffuse": diffuse,
        "counts": {
            "local": len(local),
            "long_range": len(long_range),
            "diffuse": len(diffuse),
        },
    }


@torch.no_grad()
def collect_attention(model, token_ids: torch.Tensor, device: str | None = None):
    """Run one sequence through the model and return its attention tensors."""
    device = device or next(model.parameters()).device
    model.eval()
    if token_ids.dim() == 1:
        token_ids = token_ids.unsqueeze(0)
    output = model(token_ids.to(device), return_attention=True)
    return output["attentions"]


def plot_attention_heatmaps(
    attentions: list[torch.Tensor],
    tokens: list[str],
    layers: list[int],
    heads: list[int],
    out_path,
    title: str = "",
    max_positions: int = 32,
):
    """Save a grid of query-vs-key attention heatmaps.

    The assignment asks for at least one early and one late layer with several
    heads each, labelled with the example sentence's tokens.

    Args:
        attentions: Per-layer ``(B, h, T, T)`` tensors.
        tokens: Token strings for the axis labels.
        layers: Which layers to draw.
        heads: Which heads to draw.
        out_path: Destination PNG.
        title: Figure title, typically naming the model and the sentence.
        max_positions: Truncate to this many positions so labels stay legible.
    """
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    # Axis labels are Devanagari tokens, and the default font has no glyphs for
    # them - they would render as empty boxes. Phase 1 already solved this with a
    # fallback chain that puts a Latin font first and Indic faces after it.
    from ..stats.plots import configure_fonts

    configure_fonts()

    # Tick labels are SentencePiece pieces, which mix three character sets:
    # Devanagari, Latin (in loanwords and the piece marker), and the U+2581 block
    # character SentencePiece uses to mark word starts. No single installed face
    # covers all three, so the labels take an ordered *list* of families and each
    # character falls through to whichever face has its glyph. Naming a single
    # Devanagari font here instead turns every Latin character into a box.
    import matplotlib.font_manager as font_manager

    available = {font.name for font in font_manager.fontManager.ttflist}

    # Hosted notebook images often ship no Devanagari face at all, in which case
    # every label renders as an empty box and the figure is useless. Fetch one
    # and register it rather than producing an illegible plot.
    if not any("Devanagari" in name for name in available):
        try:
            import urllib.request

            url = (
                "https://github.com/googlefonts/noto-fonts/raw/main/hinted/ttf/"
                "NotoSansDevanagari/NotoSansDevanagari-Regular.ttf"
            )
            font_path = Path("/tmp/NotoSansDevanagari-Regular.ttf")
            if not font_path.exists():
                urllib.request.urlretrieve(url, font_path)
            font_manager.fontManager.addfont(str(font_path))
            available = {font.name for font in font_manager.fontManager.ttflist}
            print(f"  installed Devanagari font for plot labels: {font_path.name}")
        except Exception as error:
            # Not fatal: the numbers are unaffected, only the tick labels.
            print(f"  note: no Devanagari font available, labels may not render ({error})")

    indic_font = [
        name
        for name in (
            "DejaVu Sans",              # Latin, punctuation, and U+2581
            "Noto Sans Devanagari",     # Devanagari
            "Noto Serif Devanagari",
        )
        if name in available
    ] or ["DejaVu Sans"]

    limit = min(max_positions, len(tokens), attentions[0].shape[-1])
    labels = tokens[:limit]

    figure, axes = plt.subplots(
        len(layers), len(heads), figsize=(4 * len(heads), 3.6 * len(layers)), squeeze=False
    )

    for row, layer in enumerate(layers):
        for column, head in enumerate(heads):
            axis = axes[row][column]
            weights = attentions[layer][0, head, :limit, :limit].float().cpu().numpy()

            image = axis.imshow(weights, cmap="viridis", aspect="auto", vmin=0.0)
            axis.set_title(f"Layer {layer}, head {head}", fontsize=10)
            axis.set_xlabel("key position")
            axis.set_ylabel("query position")

            # Label every other position: a full set overlaps illegibly, and the
            # point of the labels is to show *which* tokens a head favours.
            #
            # The Devanagari font is named explicitly here rather than relying on
            # the rcParams family chain: matplotlib resolves tick labels through a
            # separate path that does not fall through the chain, so the glyphs
            # would render as empty boxes despite the font being installed.
            step = max(1, limit // 16)
            axis.set_xticks(range(0, limit, step))
            axis.set_xticklabels(
                labels[::step], rotation=90, fontsize=6, fontfamily=indic_font
            )
            axis.set_yticks(range(0, limit, step))
            axis.set_yticklabels(labels[::step], fontsize=6, fontfamily=indic_font)

            figure.colorbar(image, ax=axis, fraction=0.046, label="attention weight")

    if title:
        figure.suptitle(title, fontsize=12)
    figure.tight_layout()
    figure.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    return out_path


def plot_head_summary(stats: dict, out_path, title: str = ""):
    """Save the per-head entropy and mean-distance maps as one figure."""
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    import numpy as np

    from ..stats.plots import configure_fonts

    configure_fonts()

    entropy = np.array(stats["normalized_entropy"])
    distance = np.array(stats["mean_distance"])

    figure, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    for axis, data, name, cmap in (
        (axes[0], entropy, "Normalized attention entropy", "magma"),
        (axes[1], distance, "Mean attention distance (positions)", "viridis"),
    ):
        image = axis.imshow(data, cmap=cmap, aspect="auto")
        axis.set_title(name)
        axis.set_xlabel("head")
        axis.set_ylabel("layer")
        axis.set_xticks(range(data.shape[1]))
        axis.set_yticks(range(data.shape[0]))
        figure.colorbar(image, ax=axis, fraction=0.046)

    if title:
        figure.suptitle(title, fontsize=12)
    figure.tight_layout()
    figure.savefig(out_path, dpi=150, bbox_inches="tight")
    plt.close(figure)
    return out_path

In [ ]:
%%writefile /kaggle/working/src/lma/eval/generation.py
"""Generation quality: BLEU, chrF, ROUGE-L, and diversity diagnostics.

The three overlap metrics are implemented here rather than pulled from a library
so that their behaviour on Devanagari is explicit and inspectable. The usual
implementations tokenize on whitespace and Latin punctuation, which is adequate
for Hindi and Nepali - both space-separated - but chrF in particular is defined
over characters, and getting that right for a script with combining marks matters
more than the convenience of an import.

A caveat worth stating up front, and worth repeating in the report: these are
similarity metrics built for tasks with a correct answer, such as translation or
summarization. Open-ended continuation has no single right answer, so a fluent,
sensible continuation that simply differs from the held-out text scores near
zero. They are reported because the assignment asks for them, and they are
informative *comparatively* - between temperatures, and between Model H and Model
L - but their absolute values should not be read as quality.
"""

from __future__ import annotations

import math
from collections import Counter


def _ngrams(tokens: list[str], n: int) -> Counter:
    """Count n-grams in a token list."""
    return Counter(tuple(tokens[i : i + n]) for i in range(len(tokens) - n + 1))


def corpus_bleu(
    hypotheses: list[str], references: list[str], max_n: int = 4
) -> dict:
    """Corpus-level BLEU-4 with the standard brevity penalty.

    Corpus-level rather than sentence-level: precision numerators and
    denominators are pooled across the whole corpus before dividing, which is the
    standard definition and far more stable than averaging per-sentence scores
    when individual hypotheses may share no 4-gram with their reference.

    Args:
        hypotheses: Generated continuations.
        references: Reference continuations, aligned with `hypotheses`.
        max_n: Maximum n-gram order. 4 is conventional; the report states it.

    Returns:
        Dict with ``bleu`` (0-100), the per-order precisions, and the brevity
        penalty.
    """
    if len(hypotheses) != len(references):
        raise ValueError("hypotheses and references must be aligned")

    matches = [0] * max_n
    totals = [0] * max_n
    hyp_length = 0
    ref_length = 0

    for hyp, ref in zip(hypotheses, references):
        hyp_tokens = hyp.split()
        ref_tokens = ref.split()
        hyp_length += len(hyp_tokens)
        ref_length += len(ref_tokens)

        for order in range(1, max_n + 1):
            hyp_ngrams = _ngrams(hyp_tokens, order)
            ref_ngrams = _ngrams(ref_tokens, order)
            # Clipped counts: an n-gram repeated in the hypothesis can only match
            # as often as it appears in the reference, which is what stops a
            # degenerate repetitive output from scoring highly.
            matches[order - 1] += sum(
                min(count, ref_ngrams[gram]) for gram, count in hyp_ngrams.items()
            )
            totals[order - 1] += max(sum(hyp_ngrams.values()), 0)

    precisions = []
    for matched, total in zip(matches, totals):
        precisions.append(matched / total if total else 0.0)

    # Geometric mean of the precisions. Any zero-precision order sends BLEU to
    # zero, which is common for open-ended generation at order 4.
    if all(p > 0 for p in precisions):
        log_mean = sum(math.log(p) for p in precisions) / max_n
        geometric = math.exp(log_mean)
    else:
        geometric = 0.0

    # Brevity penalty: without it, emitting a single high-precision word would
    # score perfectly.
    if hyp_length == 0:
        brevity = 0.0
    elif hyp_length > ref_length:
        brevity = 1.0
    else:
        brevity = math.exp(1 - ref_length / hyp_length)

    return {
        "bleu": 100 * brevity * geometric,
        "precisions": [100 * p for p in precisions],
        "brevity_penalty": brevity,
        "hypothesis_length": hyp_length,
        "reference_length": ref_length,
    }


def chrf_score(
    hypotheses: list[str], references: list[str], max_n: int = 6, beta: float = 2.0
) -> dict:
    """Corpus-level chrF: F-score over character n-grams.

    Character n-grams rather than word n-grams, which suits morphologically rich
    languages: Hindi and Nepali inflect heavily, so two forms of the same lemma
    share most of their characters but count as entirely different words under
    BLEU. chrF therefore gives partial credit where BLEU gives none, and is the
    more informative of the two here.

    `beta` weights recall over precision; 2.0 is the standard chrF setting.
    """
    if len(hypotheses) != len(references):
        raise ValueError("hypotheses and references must be aligned")

    total_precision = 0.0
    total_recall = 0.0
    orders_counted = 0

    for order in range(1, max_n + 1):
        matches = 0
        hyp_total = 0
        ref_total = 0

        for hyp, ref in zip(hypotheses, references):
            # Whitespace is removed so chrF measures the characters of the text
            # rather than its spacing, per the standard definition.
            hyp_chars = list(hyp.replace(" ", ""))
            ref_chars = list(ref.replace(" ", ""))

            hyp_ngrams = _ngrams(hyp_chars, order)
            ref_ngrams = _ngrams(ref_chars, order)

            matches += sum(min(c, ref_ngrams[g]) for g, c in hyp_ngrams.items())
            hyp_total += sum(hyp_ngrams.values())
            ref_total += sum(ref_ngrams.values())

        if hyp_total and ref_total:
            total_precision += matches / hyp_total
            total_recall += matches / ref_total
            orders_counted += 1

    if not orders_counted:
        return {"chrf": 0.0, "precision": 0.0, "recall": 0.0}

    precision = total_precision / orders_counted
    recall = total_recall / orders_counted

    if precision + recall == 0:
        f_score = 0.0
    else:
        beta_sq = beta**2
        f_score = (1 + beta_sq) * precision * recall / (beta_sq * precision + recall)

    return {
        "chrf": 100 * f_score,
        "precision": 100 * precision,
        "recall": 100 * recall,
        "beta": beta,
    }


def _lcs_length(a: list[str], b: list[str]) -> int:
    """Longest common subsequence length, computed in O(len(a) * len(b)) time.

    Only the previous row is kept, so memory is linear rather than quadratic -
    which matters when comparing continuations hundreds of tokens long.
    """
    if not a or not b:
        return 0

    previous = [0] * (len(b) + 1)
    for token_a in a:
        current = [0]
        for j, token_b in enumerate(b):
            if token_a == token_b:
                current.append(previous[j] + 1)
            else:
                current.append(max(current[j], previous[j + 1]))
        previous = current
    return previous[-1]


def rouge_l(hypotheses: list[str], references: list[str], beta: float = 1.2) -> dict:
    """Corpus-level ROUGE-L, the F-measure over longest common subsequences.

    Unlike BLEU's contiguous n-grams, the LCS rewards words appearing in the
    right *order* without requiring adjacency, so it tolerates the insertions and
    reorderings that a fluent but different continuation naturally contains.
    """
    if len(hypotheses) != len(references):
        raise ValueError("hypotheses and references must be aligned")

    f_scores = []
    for hyp, ref in zip(hypotheses, references):
        hyp_tokens = hyp.split()
        ref_tokens = ref.split()
        if not hyp_tokens or not ref_tokens:
            f_scores.append(0.0)
            continue

        lcs = _lcs_length(hyp_tokens, ref_tokens)
        if lcs == 0:
            f_scores.append(0.0)
            continue

        precision = lcs / len(hyp_tokens)
        recall = lcs / len(ref_tokens)
        beta_sq = beta**2
        f_scores.append(
            (1 + beta_sq) * precision * recall / (recall + beta_sq * precision)
        )

    mean = sum(f_scores) / max(len(f_scores), 1)
    return {"rouge_l": 100 * mean, "samples": len(f_scores)}


def diversity_metrics(texts: list[str], max_n: int = 4) -> dict:
    """Repetition rate and Distinct-1/2 over generated text.

    These need no reference, which makes them the more honest diagnostics for
    open-ended generation:

    * **Repetition rate** - the fraction of n-grams that are not unique. A model
      stuck in a loop scores near 1; varied text scores low.
    * **Distinct-1 / Distinct-2** - unique unigrams and bigrams as a fraction of
      all produced. Falls as a model collapses onto a few frequent tokens, which
      is the characteristic failure of greedy decoding and low temperature.
    """
    all_tokens: list[str] = []
    per_text_repetition = []

    for text in texts:
        tokens = text.split()
        all_tokens.extend(tokens)

        if len(tokens) >= max_n:
            grams = _ngrams(tokens, max_n)
            total = sum(grams.values())
            unique = len(grams)
            per_text_repetition.append(1 - unique / total if total else 0.0)

    result = {
        "repetition_rate": (
            sum(per_text_repetition) / len(per_text_repetition)
            if per_text_repetition
            else 0.0
        ),
        "repetition_ngram_order": max_n,
        "total_tokens": len(all_tokens),
    }

    for order, name in ((1, "distinct_1"), (2, "distinct_2")):
        grams = _ngrams(all_tokens, order) if len(all_tokens) >= order else Counter()
        total = sum(grams.values())
        result[name] = len(grams) / total if total else 0.0

    return result


def evaluate_generations(
    hypotheses: list[str], references: list[str]
) -> dict:
    """Run every generation metric over one set of continuations."""
    return {
        **corpus_bleu(hypotheses, references),
        **chrf_score(hypotheses, references),
        **rouge_l(hypotheses, references),
        **diversity_metrics(hypotheses),
    }

In [ ]:
%%writefile /kaggle/working/src/lma/eval/lm_metrics.py
"""Intrinsic language-modeling metrics: cross-entropy, perplexity, bits-per-byte.

Perplexity is the natural measure of a language model's fit, but it is not
comparable across models whose tokenizers differ: a model whose tokenizer splits
text into fewer, longer pieces faces fewer prediction decisions per sentence and
scores better without being better. Bits-per-byte removes that advantage by
re-expressing the same total loss per byte of the original UTF-8 text, which is a
property of the text rather than of the segmentation. Both are reported for both
models, as the assignment requires.
"""

from __future__ import annotations

import math

import torch

from ..train.data import TokenBinaryDataset


@torch.no_grad()
def evaluate_language_model(
    model,
    dataset: TokenBinaryDataset,
    batch_size: int = 16,
    max_batches: int | None = None,
    device: str | None = None,
    tokenizer=None,
) -> dict:
    """Cross-entropy, perplexity, and bits-per-byte on a held-out split.

    The split is walked sequentially rather than sampled, so the result is a
    property of the data and not of a random draw.

    Args:
        model: A trained :class:`~lma.model.TransformerLM`.
        dataset: Sampler over the held-out token binary.
        batch_size: Sequences per forward pass.
        max_batches: Stop after this many batches. ``None`` walks the whole split.
        device: Where to run. Defaults to the model's device.
        tokenizer: SentencePiece processor. Required for bits-per-byte, which
            needs the UTF-8 length of the text the tokens decode to.

    Returns:
        Dict with ``cross_entropy`` (nats/token), ``perplexity``, ``tokens``,
        and - when a tokenizer is supplied - ``bits_per_byte`` and ``bytes``.
    """
    device = device or next(model.parameters()).device
    model.eval()

    total_nats = 0.0
    total_tokens = 0
    total_bytes = 0

    for x, y in dataset.sequential_batches(batch_size, limit=max_batches, device=str(device)):
        logits = model(x)["logits"]

        # Sum rather than mean: batches can differ in size, and a mean of means
        # would silently weight a short final batch as heavily as a full one.
        loss = torch.nn.functional.cross_entropy(
            logits.reshape(-1, logits.size(-1)),
            y.reshape(-1),
            reduction="sum",
        )
        total_nats += loss.float().item()
        total_tokens += y.numel()

        if tokenizer is not None:
            # Bits-per-byte needs the byte length of the text these targets
            # represent, decoded with the model's own tokenizer.
            for row in y.tolist():
                total_bytes += len(tokenizer.decode(row).encode("utf-8"))

    cross_entropy = total_nats / max(total_tokens, 1)
    result = {
        "cross_entropy": cross_entropy,
        "perplexity": math.exp(min(cross_entropy, 20)),
        "tokens": total_tokens,
    }

    if tokenizer is not None and total_bytes:
        # Total loss in nats -> bits, spread over the bytes of the source text.
        # log(2) converts nats to bits; dividing by bytes rather than tokens is
        # what makes the figure comparable between different tokenizers.
        result["bits_per_byte"] = total_nats / (total_bytes * math.log(2))
        result["bytes"] = total_bytes
        result["bytes_per_token"] = total_bytes / max(total_tokens, 1)

    return result


def compare_models(results: dict[str, dict]) -> str:
    """Format a side-by-side table of per-model metrics for the report."""
    languages = list(results)
    header = f"| Metric | {' | '.join(languages)} |"
    rule = "|---|" + "---|" * len(languages)

    rows = [header, rule]
    for label, key, fmt in (
        ("Cross-entropy (nats/token)", "cross_entropy", "{:.4f}"),
        ("Perplexity", "perplexity", "{:.2f}"),
        ("Bits-per-byte", "bits_per_byte", "{:.4f}"),
        ("Held-out tokens", "tokens", "{:,}"),
    ):
        cells = []
        for lang in languages:
            value = results[lang].get(key)
            cells.append(fmt.format(value) if value is not None else "-")
        rows.append(f"| {label} | {' | '.join(cells)} |")

    return "\n".join(rows)

In [ ]:
%%writefile /kaggle/working/src/lma/stats/plots.py
"""Plotting for the Phase 1 report.

Kept strictly separate from computation: every function here takes already-computed
statistics dicts and returns a saved figure path. Nothing in this module reads a
corpus or trains a tokenizer.

The project requires each plot to carry a title, axis labels, and a legend where
applicable, so :func:`_finalize` applies those uniformly and every function goes
through it.
"""

from __future__ import annotations

from pathlib import Path
from typing import Sequence

import matplotlib

matplotlib.use("Agg")  # Headless: figures are written to disk, never displayed.
import matplotlib.pyplot as plt

#: One consistent colour per language across every figure in the report.
LANGUAGE_COLORS = {"hi": "#1f77b4", "ne": "#d62728", "as": "#9467bd", "bn": "#2ca02c"}
LANGUAGE_LABELS = {
    "hi": "Hindi (Model H)",
    "ne": "Nepali (Model L)",
    "as": "Assamese",
    "bn": "Bengali",
}

#: Fonts covering the Indic scripts used here. Devanagari and Bengali need
#: different faces, and figures may mix both, so all are offered to the
#: fallback chain and matplotlib picks per glyph.
_INDIC_FONTS = (
    "Noto Sans Devanagari",
    "Noto Serif Devanagari",
    "Lohit Devanagari",
    "Samyak Devanagari",
    "Noto Sans Bengali",
    "Noto Serif Bengali",
    "Lohit Bengali",
    "Mukti Narrow",
    "FreeSans",
)

#: Latin-capable fallbacks. Required because the Noto Indic faces contain *no*
#: Latin glyphs at all - every English axis label, legend entry, and numeral would
#: render as a tofu box if an Indic font were used alone.
_LATIN_FONTS = ("DejaVu Sans", "Liberation Sans", "FreeSans", "Arial")


def configure_fonts() -> str:
    """Install a font fallback chain that renders both Indic and Latin text.

    Matplotlib resolves ``font.family`` as an ordered list, falling through to the
    next entry per missing glyph. Figures here mix Indic tick labels with English
    axis titles, and no single available font covers both: DejaVu Sans has no
    Indic coverage, and the Noto Indic faces have no Latin coverage. Listing a
    Latin font first and the Indic faces after it means Latin text uses the Latin
    font and Indic characters fall through to whichever face covers them.

    Returns:
        A human-readable description of the resolved chain, for logging.
    """
    from matplotlib import font_manager

    available = {font.name for font in font_manager.fontManager.ttflist}
    latin = [name for name in _LATIN_FONTS if name in available]
    indic = [name for name in _INDIC_FONTS if name in available]

    chain = latin[:1] + indic + latin[1:]
    if not chain:
        return plt.rcParams["font.family"][0]

    plt.rcParams["font.family"] = "sans-serif"
    plt.rcParams["font.sans-serif"] = chain
    # Minus signs come from the Indic font otherwise, which lacks U+2212.
    plt.rcParams["axes.unicode_minus"] = False
    return " -> ".join(chain) if indic else f"{chain[0]} (no Indic font found)"


def _finalize(fig, ax, title: str, xlabel: str, ylabel: str, output_path: Path, legend: bool = False) -> Path:
    """Apply required labelling, save the figure, and close it.

    Args:
        fig: Matplotlib figure.
        ax: Primary axes.
        title: Figure title.
        xlabel: X-axis label.
        ylabel: Y-axis label.
        output_path: Destination image path.
        legend: Whether to draw a legend.

    Returns:
        The path the figure was written to.
    """
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.grid(True, alpha=0.3, linestyle="--")
    if legend:
        ax.legend(fontsize=9)
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def plot_corpus_size_comparison(stats_by_language: dict[str, dict], output_path: Path) -> Path:
    """Grouped bars comparing documents, words, and characters across languages."""
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    metrics = [("documents", "Documents"), ("words", "Words"), ("characters", "Characters")]

    for ax, (key, label) in zip(axes, metrics):
        languages = list(stats_by_language)
        values = [stats_by_language[lang]["totals"][key] for lang in languages]
        colors = [LANGUAGE_COLORS.get(lang, "#7f7f7f") for lang in languages]
        bars = ax.bar([LANGUAGE_LABELS.get(l, l) for l in languages], values, color=colors)
        for bar, value in zip(bars, values):
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{value:,.0f}",
                ha="center",
                va="bottom",
                fontsize=8,
            )
        ax.set_title(label, fontsize=11, fontweight="bold")
        ax.set_xlabel("Model / language", fontsize=9)
        ax.set_ylabel(f"{label} (count)", fontsize=9)
        ax.set_yscale("log")
        ax.grid(True, alpha=0.3, linestyle="--", axis="y")
        ax.tick_params(axis="x", labelsize=8)

    fig.suptitle("Corpus size comparison: Hindi (higher-resource) vs Nepali (lower-resource)",
                 fontsize=12, fontweight="bold")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def plot_document_length_distribution(stats_by_language: dict[str, dict], output_path: Path) -> Path:
    """Quantile curves of document length in words, one line per language."""
    fig, ax = plt.subplots(figsize=(8, 5))
    quantile_keys = ["min", "p10", "p25", "median", "p75", "p90", "p99", "max"]
    positions = [0, 10, 25, 50, 75, 90, 99, 100]

    for language, stats in stats_by_language.items():
        lengths = stats["document_length_words"]
        values = [lengths[key] for key in quantile_keys]
        ax.plot(
            positions,
            values,
            marker="o",
            label=LANGUAGE_LABELS.get(language, language),
            color=LANGUAGE_COLORS.get(language, "#7f7f7f"),
        )

    ax.set_yscale("log")
    return _finalize(
        fig, ax,
        title="Document length distribution by quantile",
        xlabel="Quantile (percentile of documents)",
        ylabel="Document length (words, log scale)",
        output_path=output_path,
        legend=True,
    )


def plot_source_composition(stats: dict, language: str, output_path: Path) -> Path:
    """Horizontal bars of word counts per source, coloured by collection method."""
    by_source = stats["by_source"]["words"]
    if not by_source:
        by_source = {"(none)": 0}
    sources = list(by_source)[:15][::-1]
    values = [by_source[s] for s in sources]

    fig, ax = plt.subplots(figsize=(9, max(3.0, 0.45 * len(sources) + 1.5)))
    bars = ax.barh(sources, values, color=LANGUAGE_COLORS.get(language, "#7f7f7f"))
    for bar, value in zip(bars, values):
        ax.text(bar.get_width(), bar.get_y() + bar.get_height() / 2, f" {value:,}", va="center", fontsize=8)

    ax.set_xscale("log")
    return _finalize(
        fig, ax,
        title=f"Corpus composition by source - {LANGUAGE_LABELS.get(language, language)}",
        xlabel="Words contributed (log scale)",
        ylabel="Source identifier",
        output_path=output_path,
    )


def plot_manual_fraction(stats_by_language: dict[str, dict], output_path: Path, requirement: float = 0.20) -> Path:
    """Stacked bars of manual vs downloaded token share against the 20% requirement."""
    languages = list(stats_by_language)
    manual = []
    downloaded = []
    for language in languages:
        words = stats_by_language[language]["collection_split"]["words"]
        total = sum(words.values()) or 1
        manual.append(words.get("manual", 0) / total * 100)
        downloaded.append(sum(v for k, v in words.items() if k != "manual") / total * 100)

    labels = [LANGUAGE_LABELS.get(l, l) for l in languages]
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.bar(labels, manual, label="Manually collected", color="#2ca02c")
    ax.bar(labels, downloaded, bottom=manual, label="Downloaded (public corpora)", color="#aec7e8")
    ax.axhline(
        requirement * 100,
        color="#d62728",
        linestyle="--",
        linewidth=1.8,
        label=f"Requirement: {requirement:.0%} manual",
    )

    for index, value in enumerate(manual):
        ax.text(index, value / 2, f"{value:.1f}%", ha="center", va="center", fontsize=9, fontweight="bold")

    ax.set_ylim(0, 100)
    return _finalize(
        fig, ax,
        title="Manual vs downloaded share of training tokens",
        xlabel="Model / language",
        ylabel="Share of training words (%)",
        output_path=output_path,
        legend=True,
    )


def plot_cleaning_funnel(cleaning_stats: dict[str, dict], output_path: Path) -> Path:
    """Bars showing how many documents each filter rejected, per language."""
    languages = list(cleaning_stats)
    reasons: list[str] = []
    for language in languages:
        for reason in cleaning_stats[language].get("rejected", {}):
            if reason not in reasons:
                reasons.append(reason)
    reasons.sort(key=lambda r: -sum(cleaning_stats[l].get("rejected", {}).get(r, 0) for l in languages))
    reasons = reasons[:12]

    fig, ax = plt.subplots(figsize=(10, 5.5))
    width = 0.8 / max(1, len(languages))
    positions = range(len(reasons))

    for index, language in enumerate(languages):
        rejected = cleaning_stats[language].get("rejected", {})
        values = [rejected.get(reason, 0) for reason in reasons]
        offsets = [p + index * width for p in positions]
        ax.bar(
            offsets, values, width=width,
            label=LANGUAGE_LABELS.get(language, language),
            color=LANGUAGE_COLORS.get(language, "#7f7f7f"),
        )

    ax.set_xticks([p + width * (len(languages) - 1) / 2 for p in positions])
    ax.set_xticklabels(reasons, rotation=40, ha="right", fontsize=8)
    ax.set_yscale("symlog")
    return _finalize(
        fig, ax,
        title="Documents rejected by each cleaning filter",
        xlabel="Rejection reason (first filter to fire)",
        ylabel="Documents rejected (symlog scale)",
        output_path=output_path,
        legend=True,
    )


def plot_vocab_sweep(sweep_by_language: dict[str, Sequence[dict]], output_path: Path) -> Path:
    """Fertility and byte-fallback rate against vocabulary size for every candidate."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for language, results in sweep_by_language.items():
        base_color = LANGUAGE_COLORS.get(language, "#7f7f7f")
        for model_type, marker, style in (("unigram", "o", "-"), ("bpe", "s", "--")):
            points = sorted(
                (r for r in results if "error" not in r and r.get("model_type") == model_type),
                key=lambda r: r["vocab_size"],
            )
            if not points:
                continue
            sizes = [r["vocab_size"] for r in points]
            label = f"{LANGUAGE_LABELS.get(language, language)} - {model_type}"
            axes[0].plot(sizes, [r["fertility"] for r in points],
                         marker=marker, linestyle=style, color=base_color, label=label)
            # Prefer the script-only rate: the raw rate is dominated by newline
            # byte tokens, which measure document formatting rather than coverage.
            axes[1].plot(
                sizes,
                [r.get("script_byte_fallback_rate", r["byte_fallback_rate"]) * 100 for r in points],
                marker=marker, linestyle=style, color=base_color, label=label,
            )

    for ax, title, ylabel in (
        (axes[0], "Fertility vs vocabulary size", "Fertility (tokens per word)"),
        (axes[1], "Script byte-fallback vs vocabulary size", "Non-whitespace byte tokens (% of tokens)"),
    ):
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.set_xlabel("Vocabulary size (pieces)", fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_xscale("log")
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.legend(fontsize=8)

    fig.suptitle("Tokenizer vocabulary selection on held-out text", fontsize=12, fontweight="bold")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def plot_token_frequency(freq_by_language: dict[str, dict], output_path: Path, top_k: int = 30) -> Path:
    """Zipf-style log-log rank/frequency curves plus cumulative coverage."""
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    for language, freq in freq_by_language.items():
        color = LANGUAGE_COLORS.get(language, "#7f7f7f")
        label = LANGUAGE_LABELS.get(language, language)
        tokens = freq.get("top_tokens", [])
        if tokens:
            ranks = range(1, len(tokens) + 1)
            axes[0].plot(list(ranks), [t["share"] * 100 for t in tokens], marker=".", color=color, label=label)

        coverage = freq.get("coverage", {})
        if coverage:
            points = sorted((int(k.split("_")[1]), v) for k, v in coverage.items())
            axes[1].plot([p[0] for p in points], [p[1] * 100 for p in points],
                         marker="o", color=color, label=label)

    axes[0].set_xscale("log")
    axes[0].set_yscale("log")
    axes[0].set_title("Token frequency by rank (held-out text)", fontsize=11, fontweight="bold")
    axes[0].set_xlabel("Token rank (log scale)", fontsize=10)
    axes[0].set_ylabel("Share of all tokens (%, log scale)", fontsize=10)

    axes[1].set_xscale("log")
    axes[1].set_title("Cumulative coverage by vocabulary prefix", fontsize=11, fontweight="bold")
    axes[1].set_xlabel("Top-N most frequent tokens (log scale)", fontsize=10)
    axes[1].set_ylabel("Cumulative share of tokens (%)", fontsize=10)

    for ax in axes:
        ax.grid(True, alpha=0.3, linestyle="--")
        ax.legend(fontsize=9)

    fig.suptitle("Token-frequency statistics per language", fontsize=12, fontweight="bold")
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    return output_path


def plot_piece_length_histogram(piece_stats_by_language: dict[str, dict], output_path: Path) -> Path:
    """Grouped bars of vocabulary piece lengths in characters."""
    fig, ax = plt.subplots(figsize=(9, 5))
    keys = [str(i) for i in range(1, 10)] + ["10+"]
    languages = list(piece_stats_by_language)
    width = 0.8 / max(1, len(languages))

    for index, language in enumerate(languages):
        histogram = piece_stats_by_language[language].get("piece_length_histogram", {})
        total = sum(histogram.values()) or 1
        values = [histogram.get(key, 0) / total * 100 for key in keys]
        offsets = [i + index * width for i in range(len(keys))]
        ax.bar(offsets, values, width=width,
               label=LANGUAGE_LABELS.get(language, language),
               color=LANGUAGE_COLORS.get(language, "#7f7f7f"))

    ax.set_xticks([i + width * (len(languages) - 1) / 2 for i in range(len(keys))])
    ax.set_xticklabels(keys)
    return _finalize(
        fig, ax,
        title="Vocabulary piece-length distribution",
        xlabel="Piece length (characters, excluding word-boundary marker)",
        ylabel="Share of vocabulary (%)",
        output_path=output_path,
        legend=True,
    )


def plot_split_sizes(split_stats_by_language: dict[str, dict], output_path: Path, unit: str = "characters") -> Path:
    """Grouped bars of split sizes per train/validation/test split.

    Args:
        split_stats_by_language: ``{language: {split: {"totals": {"words": n}}}}``.
            The ``words`` key carries whatever quantity `unit` names.
        output_path: Destination image path.
        unit: Name of the plotted quantity, used in the axis label so the figure
            never misstates what it shows.
    """
    splits = ["train", "validation", "test"]
    fig, ax = plt.subplots(figsize=(9, 5))
    languages = list(split_stats_by_language)
    width = 0.8 / max(1, len(languages))

    for index, language in enumerate(languages):
        per_split = split_stats_by_language[language]
        values = [per_split.get(split, {}).get("totals", {}).get("words", 0) for split in splits]
        offsets = [i + index * width for i in range(len(splits))]
        bars = ax.bar(offsets, values, width=width,
                      label=LANGUAGE_LABELS.get(language, language),
                      color=LANGUAGE_COLORS.get(language, "#7f7f7f"))
        for bar, value in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{value:,}",
                    ha="center", va="bottom", fontsize=7, rotation=90)

    ax.set_xticks([i + width * (len(languages) - 1) / 2 for i in range(len(splits))])
    ax.set_xticklabels([s.capitalize() for s in splits])
    ax.set_yscale("log")
    return _finalize(
        fig, ax,
        title="Train / validation / test split sizes",
        xlabel="Split",
        ylabel=f"{unit.capitalize()} in split (log scale)",
        output_path=output_path,
        legend=True,
    )

In [ ]:
%%writefile /kaggle/working/src/lma/reasoning/__init__.py
"""Phase 3: synthetic comparative-reasoning data, finetuning, and evaluation.

The dataset is generated programmatically rather than downloaded, as the
assignment requires, so every ground-truth label is known by construction and the
whole corpus is reproducible from a seed.
"""

from .templates import TEMPLATES, Dimension, LanguageTemplates, get_templates

__all__ = ["TEMPLATES", "Dimension", "LanguageTemplates", "get_templates"]

In [ ]:
%%writefile /kaggle/working/src/lma/reasoning/templates.py
"""Devanagari surface forms for the synthetic comparative-reasoning task.

Everything language-specific lives here, separated from the generation logic in
:mod:`lma.reasoning.generate`, so the two languages are configured rather than
special-cased and a third language could be added without touching the sampler.

Why the wording was chosen the way it was
-----------------------------------------
The assignment requires the finetuning text to be "in the target language's
script and natural phrasing for that language (not English prompts with foreign
names only)". Three constraints shaped every string below, and all three were
measured against this project's own Phase 1 tokenizers rather than assumed:

1. **No byte fallback.** A piece the vocabulary does not cover is encoded as raw
   UTF-8 bytes, which makes an answer span several unpredictable tokens and
   turns exact-match scoring into guesswork. Measured: ``उत्तर:`` byte-falls-back
   in Nepali because that vocabulary has no ``:`` piece, while ``उत्तर -`` is
   clean in both languages. Hence :data:`ANSWER_PREFIX` uses the dash.

2. **Single-token answers.** Every entity name below encodes to exactly one
   ``▁``-prefixed piece in its language's tokenizer, so the answer is one token.
   That is what lets the finetuning loss be masked to the answer span exactly,
   and what makes exact match unambiguous. Numbers were rejected as answers for
   the opposite reason: ``split_digits: true`` means ``12`` becomes ``▁ 1 2``,
   three tokens whose partial emission would score as a wrong answer.

3. **Grammatical authenticity.** The comparative constructions differ
   structurally between the two languages, and using one language's frame with
   the other's words would produce text the project's own language filter in
   ``lma/data/language_id.py`` would reject. Hindi compares with the postposition
   ``से`` and the copula ``है``; Nepali suffixes ``भन्दा`` directly onto the
   standard of comparison and uses the copula ``छ``. These are the same forms
   that filter relies on to tell the two languages apart.
"""

from __future__ import annotations

from dataclasses import dataclass


@dataclass(frozen=True)
class Dimension:
    """One comparable attribute, with both poles of its comparison.

    A dimension supplies the adjective used to state a premise and the two
    superlatives used to ask a question, so that "taller" premises can be
    queried for either the tallest or the shortest entity.

    Attributes:
        key: Stable identifier used in the dataset record and in statistics.
        premise_more: Adjective phrase asserting the subject exceeds the
            standard, e.g. Hindi ``लंबा`` (tall) in "A is taller than B".
        question_most: Superlative asking for the maximum, e.g. "who is tallest".
        question_least: Superlative asking for the minimum.
    """

    key: str
    premise_more: str
    question_most: str
    question_least: str


@dataclass(frozen=True)
class LanguageTemplates:
    """Every surface string needed to render one language's examples.

    Attributes:
        language: ISO code, ``"hi"`` or ``"ne"``.
        entities: Single-token personal names used as the compared entities.
        dimensions: Comparable attributes available to the sampler.
        premise: Format string taking ``subject``, ``standard`` and ``adjective``.
        question: Format string taking ``superlative``.
        answer_prefix: Text introducing the answer, and the boundary the
            finetuning loss is masked from.
    """

    language: str
    entities: tuple[str, ...]
    dimensions: tuple[Dimension, ...]
    premise: str
    question: str
    answer_prefix: str

    def render_premise(self, subject: str, standard: str, dimension: Dimension) -> str:
        """Render "``subject`` is more ``dimension`` than ``standard``"."""
        return self.premise.format(
            subject=subject, standard=standard, adjective=dimension.premise_more
        )

    def render_question(self, dimension: Dimension, ask_most: bool) -> str:
        """Render the superlative question for one dimension."""
        superlative = dimension.question_most if ask_most else dimension.question_least
        return self.question.format(superlative=superlative)


#: Hindi (Model H). The comparative frame is "<subject> <standard> से <adjective>
#: है" - the standard of comparison takes the postposition से, and the sentence
#: closes with the copula है, both of which are Hindi markers in language_id.py.
HINDI = LanguageTemplates(
    language="hi",
    entities=(
        "राम", "श्याम", "मोहन", "सीता", "गीता", "अमित", "रवि", "विजय",
        "अनिल", "सुनील", "दीपक", "अर्जुन", "संजय", "राहुल", "कृष्ण",
        "शिव", "गणेश", "सूरज", "मनोज", "अजय", "विनोद", "रमेश", "सुरेश",
        "महेश", "राजेश", "दिनेश", "नरेश", "हरि", "गोपाल", "प्रकाश",
        "संतोष", "अशोक", "रोहित", "सचिन", "विकास", "पवन", "नितिन",
        "किरण", "लक्ष्मी", "रेखा", "माया", "राधा",
    ),
    dimensions=(
        Dimension("height", "लंबा", "सबसे लंबा", "सबसे छोटा"),
        Dimension("age", "बड़ा", "सबसे बड़ा", "सबसे छोटा"),
        Dimension("speed", "तेज़", "सबसे तेज़", "सबसे धीमा"),
        Dimension("wealth", "अमीर", "सबसे अमीर", "सबसे गरीब"),
    ),
    premise="{subject} {standard} से {adjective} है।",
    question="{superlative} कौन है?",
    answer_prefix="उत्तर -",
)

#: Nepali (Model L). The comparative frame suffixes भन्दा directly onto the
#: standard ("श्यामभन्दा अग्लो छ"), and closes with the copula छ. Both are Nepali
#: markers in language_id.py, so generated text is positively identified as
#: Nepali rather than merely "Devanagari".
NEPALI = LanguageTemplates(
    language="ne",
    entities=(
        "राम", "श्याम", "हरि", "सीता", "विनोद", "रमेश", "प्रकाश", "कमला",
        "सुरेश", "गोपाल", "कृष्ण", "शिव", "गणेश", "विष्णु", "मोहन",
        "विजय", "अर्जुन", "महेश", "सन्तोष", "विकास", "आनन्द", "किरण",
        "लक्ष्मी", "रेखा", "माया", "बिकास", "निर्मला", "दुर्गा", "भीम",
        "कुमार", "यादव", "थापा", "शर्मा",
    ),
    dimensions=(
        Dimension("height", "अग्लो", "सबैभन्दा अग्लो", "सबैभन्दा होचो"),
        Dimension("age", "जेठो", "सबैभन्दा जेठो", "सबैभन्दा कान्छो"),
        Dimension("speed", "छिटो", "सबैभन्दा छिटो", "सबैभन्दा ढिलो"),
        Dimension("wealth", "धनी", "सबैभन्दा धनी", "सबैभन्दा गरिब"),
    ),
    premise="{subject} {standard}भन्दा {adjective} छ।",
    question="{superlative} को हो?",
    answer_prefix="उत्तर -",
)

#: Registry, so scripts select by language code exactly as the Phase 1 configs do.
TEMPLATES: dict[str, LanguageTemplates] = {"hi": HINDI, "ne": NEPALI}


def get_templates(language: str) -> LanguageTemplates:
    """Return the template set for a language code.

    Args:
        language: ``"hi"`` or ``"ne"``.

    Raises:
        KeyError: If the language has no templates defined.
    """
    if language not in TEMPLATES:
        raise KeyError(
            f"no reasoning templates for {language!r}; have {sorted(TEMPLATES)}"
        )
    return TEMPLATES[language]

In [ ]:
%%writefile /kaggle/working/src/lma/reasoning/generate.py
"""Programmatic generation of the synthetic comparative-reasoning dataset.

The assignment forbids downloading an existing reasoning benchmark as the primary
finetune set: examples must be generated so the ground-truth label is known by
construction. Everything here is therefore derived from a seed, and re-running
with the same seed reproduces the corpus byte for byte.

The task
--------
Each example states a chain of pairwise comparisons and asks for the extreme
element::

    राम श्याम से लंबा है। श्याम मोहन से लंबा है। सबसे लंबा कौन है? उत्तर - राम

Answering requires chaining the inequalities rather than pattern-matching a
single sentence, which is the "multi-hop comparison" the assignment asks for. The
answer is always a single entity name, and the entity pools in
:mod:`lma.reasoning.templates` are vetted so that name is exactly one token under
that language's own Phase 1 tokenizer - which is what makes the finetuning loss
maskable to the answer span and exact-match scoring unambiguous.

Leakage control
---------------
The property being tested is generalization over **comparison structures**, not
over vocabulary. A model should answer a chain it has never seen, built from
entities it can name; it should not be asked to emit a token it was never trained
to produce. Three mechanisms, each covering what the others do not:

1. **Disjoint chains.** The unordered set of entities in an example, together
   with its dimension, forms a *chain signature*. Signatures are partitioned
   across splits, so no evaluation example reuses a combination of entities that
   any training example compared. The model cannot recall "गणेश beat दीपक"; it
   has to derive it.

2. **Evaluation-only distractor names.** A minority of the pool appears only in
   evaluation examples, mixed in alongside familiar names. These test that the
   model handles an unfamiliar participant without making the answer itself
   unproducible - the answer is drawn from the full pool.

3. **Content-hash split assignment.** Within those constraints, assignment uses
   the same salted-hash mechanism Phase 1 used
   (:class:`lma.data.splits.SplitAssigner`), so a duplicate rendering lands in
   the same split and cannot straddle train and test.

The generator additionally rejects any prompt it has already emitted, so the
corpus contains no repeated prompts at all.

A rejected earlier design, recorded because it is the obvious one
-----------------------------------------------------------------
Reserving names for the *answer* position - so every test answer was a token
absent from all training targets - makes the task unanswerable rather than hard.
Measured on a finetuned model: **0.0000** exact match on held-out-answer examples
against **0.2600** on examples whose prompts were equally unseen but whose
answers came from the training vocabulary. The evaluation set, not the model, was
the failure. Chain-level disjointness tests the same generalization without that
defect.
"""

from __future__ import annotations

import hashlib
import json
import random
from dataclasses import dataclass, field
from pathlib import Path

from ..data.splits import SplitAssigner, SplitConfig
from .templates import Dimension, LanguageTemplates, get_templates

#: Fraction of the entity pool reserved as *distractors* that only ever appear in
#: evaluation examples. Deliberately small, and deliberately not applied to the
#: answer.
#:
#: An earlier revision reserved these names for the answer position, so every
#: test answer was a token the model had never been trained to emit. That is not
#: leakage control - it makes the task unanswerable. Measured on a finetuned
#: model: 0.0000 exact match on held-out-answer examples against 0.2600 on
#: examples using training entities with equally unseen prompts. Generalization
#: is therefore tested over *comparison structures*, not over vocabulary: the
#: evaluation entities are ones the model can produce, while the chains,
#: orderings, and prompts are disjoint from anything seen in training.
HELD_OUT_ENTITY_FRACTION = 0.20

#: Chain lengths generated, as counts of entities being compared. Two entities is
#: a single comparison; three and four require chaining, which is the reasoning
#: the task is meant to exercise.
CHAIN_LENGTHS = (2, 3, 4)


@dataclass
class ReasoningExample:
    """One generated question with its known-correct answer.

    Attributes:
        prompt: Everything the model is conditioned on, ending with the answer
            prefix. The finetuning loss is masked to exclude these tokens.
        answer: The correct entity name. Exactly one token by construction.
        text: ``prompt`` and ``answer`` joined, the sequence actually trained on.
        language: ISO code of the language this was rendered in.
        dimension: Which comparable attribute was used.
        chain_length: How many entities the comparison chain covers.
        hops: Inference steps needed; ``chain_length - 1``.
        ask_most: Whether the question asked for the maximum or the minimum.
        entities: The entities in the chain, in stated order (strongest first).
        split: Assigned split, filled in by :class:`ReasoningDatasetBuilder`.
    """

    prompt: str
    answer: str
    text: str
    language: str
    dimension: str
    chain_length: int
    hops: int
    ask_most: bool
    entities: list[str]
    split: str = ""

    def to_dict(self) -> dict:
        """Serialize for JSONL output."""
        return {
            "prompt": self.prompt,
            "answer": self.answer,
            "text": self.text,
            "language": self.language,
            "dimension": self.dimension,
            "chain_length": self.chain_length,
            "hops": self.hops,
            "ask_most": self.ask_most,
            "entities": self.entities,
            "split": self.split,
        }


@dataclass
class GeneratorConfig:
    """Controls the size and shape of a generated dataset.

    Attributes:
        language: ``"hi"`` or ``"ne"``.
        total_examples: How many unique examples to emit across all splits.
        seed: Seeds both the sampler and the split hash, so the whole corpus is
            reproducible.
        train: Fraction of examples assigned to the training split.
        validation: Fraction assigned to validation.
        test: Fraction assigned to test.
        held_out_entity_fraction: Portion of the name pool reserved for
            evaluation examples.
        chain_lengths: Chain sizes to sample from.
    """

    language: str
    total_examples: int = 20_000
    seed: int = 20260812
    train: float = 0.80
    validation: float = 0.10
    test: float = 0.10
    held_out_entity_fraction: float = HELD_OUT_ENTITY_FRACTION
    chain_lengths: tuple[int, ...] = CHAIN_LENGTHS

    def to_dict(self) -> dict:
        """Serialize the resolved configuration into the output statistics."""
        return {
            "language": self.language,
            "total_examples": self.total_examples,
            "seed": self.seed,
            "splits": {
                "train": self.train,
                "validation": self.validation,
                "test": self.test,
            },
            "held_out_entity_fraction": self.held_out_entity_fraction,
            "chain_lengths": list(self.chain_lengths),
        }


class ReasoningDatasetBuilder:
    """Generates one language's reasoning corpus.

    Args:
        config: Size, seed, and split policy.
        templates: Language surface forms. Defaults to the registered set for
            ``config.language``.
    """

    def __init__(
        self, config: GeneratorConfig, templates: LanguageTemplates | None = None
    ) -> None:
        """Prepare the entity pools, the sampler, and the split assigner."""
        self.config = config
        self.templates = templates or get_templates(config.language)
        self.rng = random.Random(config.seed)

        pool = list(self.templates.entities)
        # Reserve a minority of names as evaluation-only distractors. They are
        # *additional* participants in evaluation chains, not a separate answer
        # vocabulary: every split draws its answers from the full pool, so the
        # answer is always a token the model has been trained to emit.
        random.Random(config.seed).shuffle(pool)
        distractor_count = max(2, round(len(pool) * config.held_out_entity_fraction))
        self.eval_only_entities = tuple(sorted(pool[:distractor_count]))
        self.shared_entities = tuple(sorted(pool[distractor_count:]))

        #: Names available when building a training example.
        self.train_entities = self.shared_entities
        #: Names available when building an evaluation example - the shared pool
        #: plus the distractors, so evaluation covers both familiar and novel
        #: participants.
        self.eval_entities = tuple(sorted(self.shared_entities + self.eval_only_entities))

        if len(self.shared_entities) < max(config.chain_lengths):
            raise ValueError(
                f"shared entity pool has {len(self.shared_entities)} names, too few "
                f"for chains of {max(config.chain_lengths)}"
            )

        self.split_assigner = SplitAssigner(
            SplitConfig(
                train=config.train,
                validation=config.validation,
                test=config.test,
                seed=config.seed,
            )
        )

    def _signature(self, entities: list[str], dimension: str) -> tuple:
        """Identify a comparison structure independently of its stated order.

        Two examples that compare the same people on the same attribute share a
        signature even if the premises were shuffled differently or the question
        asked for the opposite extreme. Partitioning on this is what stops an
        evaluation example from re-testing a comparison the model was trained on.
        """
        return (dimension, frozenset(entities))

    def _signature_split(self, signature: tuple) -> str:
        """Assign a chain signature to a split, deterministically.

        Hashing the signature rather than the rendered text is the whole point:
        every example sharing a comparison structure lands in the same split, so
        no evaluation chain has a training counterpart.
        """
        dimension, entities = signature
        payload = f"{self.config.seed}:{dimension}:{'|'.join(sorted(entities))}"
        digest = hashlib.blake2b(payload.encode("utf-8"), digest_size=8).digest()
        position = int.from_bytes(digest, "big") / float(1 << 64)

        if position < self.config.train:
            return "train"
        if position < self.config.train + self.config.validation:
            return "validation"
        return "test"

    def _sample_example(self, entity_pool: tuple[str, ...]) -> ReasoningExample:
        """Render one example from a given entity pool.

        The chain is built as a strict ordering: ``entities[0]`` exceeds
        ``entities[1]``, which exceeds ``entities[2]``, and so on. Premises are
        stated in a shuffled order so the answer cannot be read off the first
        sentence, which would let the model succeed without chaining.

        Evaluation-only distractors may participate in a chain but are never
        placed at the answered end of it. They exist to test that an unfamiliar
        participant does not derail the comparison; putting one in the answer
        position would instead ask the model to emit a token it was never trained
        to produce, which is the defect described in the module docstring.
        """
        chain_length = self.rng.choice(self.config.chain_lengths)
        dimension: Dimension = self.rng.choice(self.templates.dimensions)
        ask_most = self.rng.random() < 0.5

        # Build the chain so the answered end is answerable by construction. A
        # post-hoc swap is not enough: a chain sampled entirely from distractors
        # has no position to swap in, and swapping one end can leave the other a
        # distractor for the opposite polarity.
        answerable = [n for n in entity_pool if n not in self.eval_only_entities]
        answer = self.rng.choice(answerable)

        remaining = [n for n in entity_pool if n != answer]
        others = self.rng.sample(remaining, chain_length - 1)

        # entities[0] is the strongest and entities[-1] the weakest, so the
        # answer goes to whichever end the question will ask about.
        entities = [answer, *others] if ask_most else [*others, answer]

        premises = [
            self.templates.render_premise(entities[i], entities[i + 1], dimension)
            for i in range(chain_length - 1)
        ]
        # Shuffling matters: with premises in chain order the strongest entity is
        # always the first word of the prompt, and a model could answer by copying
        # it without performing any inference at all.
        self.rng.shuffle(premises)

        question = self.templates.render_question(dimension, ask_most)
        prompt = " ".join(premises + [question, self.templates.answer_prefix])
        answer = entities[0] if ask_most else entities[-1]

        return ReasoningExample(
            prompt=prompt,
            answer=answer,
            text=f"{prompt} {answer}",
            language=self.config.language,
            dimension=dimension.key,
            chain_length=chain_length,
            hops=chain_length - 1,
            ask_most=ask_most,
            entities=entities,
        )

    def build(self) -> list[ReasoningExample]:
        """Generate the full corpus, de-duplicated and split-assigned.

        Every example is routed to a split by its *comparison structure*, so all
        examples comparing the same entities on the same dimension share a split.
        Training examples draw from the shared entity pool; evaluation examples
        may additionally include the evaluation-only distractors. Answers come
        from the full pool in every split, so the model is never asked to emit a
        token it was not trained on.

        Returns:
            Every generated example, each carrying its assigned split.
        """
        targets = {
            "train": round(self.config.total_examples * self.config.train),
            "validation": round(self.config.total_examples * self.config.validation),
            "test": round(self.config.total_examples * self.config.test),
        }
        # Absorb any rounding drift into the training split.
        targets["train"] += self.config.total_examples - sum(targets.values())

        collected: dict[str, list[ReasoningExample]] = {
            "train": [],
            "validation": [],
            "test": [],
        }
        seen_prompts: set[str] = set()

        attempts = 0
        # The pools admit far more distinct structures than any realistic target,
        # so exhausting this ceiling means the configuration is wrong.
        max_attempts = max(self.config.total_examples * 60, 20_000)

        while attempts < max_attempts and any(
            len(collected[name]) < targets[name] for name in targets
        ):
            attempts += 1

            # Decide the intended split first, then sample from the pool that
            # split is allowed to use. Sampling first would bias which structures
            # can reach the evaluation splits.
            pending = [name for name in targets if len(collected[name]) < targets[name]]
            intended = self.rng.choice(pending)
            pool = (
                self.train_entities if intended == "train" else self.eval_entities
            )

            example = self._sample_example(pool)
            if example.prompt in seen_prompts:
                continue

            signature = self._signature(example.entities, example.dimension)
            split = self._signature_split(signature)
            if len(collected[split]) >= targets[split]:
                continue
            # A structure containing a distractor can only be an evaluation
            # example; if its signature hashes to train, it is simply skipped
            # rather than forced, which would break signature disjointness.
            if split == "train" and not set(example.entities) <= set(
                self.train_entities
            ):
                continue

            seen_prompts.add(example.prompt)
            example.split = split
            collected[split].append(example)

        shortfall = {
            name: targets[name] - len(collected[name])
            for name in targets
            if len(collected[name]) < targets[name]
        }
        if shortfall:
            raise RuntimeError(
                f"could not fill splits {shortfall} from a pool of "
                f"{len(self.templates.entities)} entities; reduce total_examples"
            )

        return collected["train"] + collected["validation"] + collected["test"]


def write_jsonl(examples: list[ReasoningExample], path: Path) -> None:
    """Write examples to a JSONL file, one record per line."""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for example in examples:
            handle.write(json.dumps(example.to_dict(), ensure_ascii=False))
            handle.write("\n")


def dataset_statistics(
    examples: list[ReasoningExample], builder: ReasoningDatasetBuilder
) -> dict:
    """Summarize a generated corpus for the report.

    Reports the template variety and the leakage-control evidence the assignment
    asks for: split sizes, the entity partition, and a direct check that no
    training entity appears in an evaluation example.
    """
    by_split: dict[str, int] = {}
    by_dimension: dict[str, int] = {}
    by_chain: dict[str, int] = {}
    by_hops: dict[str, int] = {}

    for example in examples:
        by_split[example.split] = by_split.get(example.split, 0) + 1
        by_dimension[example.dimension] = by_dimension.get(example.dimension, 0) + 1
        key = str(example.chain_length)
        by_chain[key] = by_chain.get(key, 0) + 1
        hop_key = str(example.hops)
        by_hops[hop_key] = by_hops.get(hop_key, 0) + 1

    train_examples = [e for e in examples if e.split == "train"]
    eval_examples = [e for e in examples if e.split in ("validation", "test")]

    def signatures(rows: list[ReasoningExample]) -> set[tuple]:
        """Comparison structures present in a set of examples."""
        return {builder._signature(e.entities, e.dimension) for e in rows}

    train_signatures = signatures(train_examples)
    eval_signatures = signatures(eval_examples)

    train_answers = {e.answer for e in train_examples}
    eval_answers = {e.answer for e in eval_examples}
    distractors = set(builder.eval_only_entities)

    return {
        "language": builder.config.language,
        "config": builder.config.to_dict(),
        "total_examples": len(examples),
        "by_split": by_split,
        "by_dimension": by_dimension,
        "by_chain_length": by_chain,
        "by_hops": by_hops,
        "unique_prompts": len({e.prompt for e in examples}),
        "entity_pools": {
            "shared": sorted(builder.shared_entities),
            "evaluation_only_distractors": sorted(builder.eval_only_entities),
            "shared_count": len(builder.shared_entities),
            "distractor_count": len(builder.eval_only_entities),
        },
        "leakage_checks": {
            # The property that matters: no evaluation example re-tests a
            # comparison structure any training example already covered.
            "chain_signature_overlap": len(train_signatures & eval_signatures),
            "train_chain_signatures": len(train_signatures),
            "eval_chain_signatures": len(eval_signatures),
            "prompt_overlap_train_eval": len(
                {e.prompt for e in train_examples} & {e.prompt for e in eval_examples}
            ),
            "duplicate_prompts": len(examples) - len({e.prompt for e in examples}),
            # The defect the earlier design had, stated precisely. A distractor
            # in the answer position asks the model to emit a token no training
            # target contained, which is unanswerable by construction. This must
            # be zero.
            "distractor_answers_in_eval": sum(
                1 for e in eval_examples if e.answer in distractors
            ),
            "distractors_in_training_examples": sorted(
                {n for e in train_examples for n in e.entities} & distractors
            ),
            # Informational, and deliberately separate from the check above: a
            # shared-pool name that happened never to be sampled as a training
            # answer. That is sampling coverage, not a design violation - the
            # name is still one the model saw in training premises and can
            # produce - so it is reported rather than asserted to be empty.
            "eval_answers_not_sampled_as_train_answer": sorted(
                (eval_answers - train_answers) - distractors
            ),
        },
        "template_variety": {
            "dimensions": len(builder.templates.dimensions),
            "chain_lengths": list(builder.config.chain_lengths),
            "question_polarities": 2,
            "entity_pool": len(builder.templates.entities),
        },
    }

In [ ]:
%%writefile /kaggle/working/src/lma/reasoning/finetune.py
"""Supervised finetuning of a pretrained model on the reasoning dataset.

This is deliberately a separate module from :mod:`lma.train.trainer` rather than
a flag on it, because the two objectives differ in one important way.

Pretraining predicts *every* token: the corpus is one flat stream and the loss is
the mean over all positions. Finetuning here predicts only the **answer**. The
prompt states premises the model is being conditioned on, not asked to produce -
training it to generate the premises would spend capacity modelling text the task
always supplies, and would let a model lower its loss without ever learning the
comparison. So the label for every prompt position is set to ``-100``, which
``F.cross_entropy`` ignores, and the loss is computed on the answer span alone.

What is reused rather than rewritten
------------------------------------
* :func:`lma.train.trainer.build_optimizer` - the decay/no-decay split is a
  property of parameter shapes, not of the objective.
* The checkpoint payload shape from :class:`lma.train.trainer.Trainer`, so a
  finetuned checkpoint resumes exactly like a pretrained one, which the
  assignment requires ("save finetuned checkpoints in the same resume-capable
  format as pretraining").
* :class:`lma.model.TransformerLM` unchanged - no new layers, no new heads. The
  task is expressed entirely in the data.
"""

from __future__ import annotations

import json
import math
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import torch

from ..model import ModelConfig, TransformerLM
from ..tokenizer.train_tokenizer import BOS_ID, EOS_ID, PAD_ID
from ..train.trainer import build_optimizer

#: Label value ignored by the loss. Matches ``ignore_index`` in the model's
#: cross-entropy call, so masked positions contribute nothing to the gradient.
IGNORE_INDEX = -100


@dataclass
class FinetuneConfig:
    """Optimization hyperparameters for one finetuning run.

    The defaults are deliberately gentler than pretraining's. The model arrives
    already trained, the dataset is small and highly regular, and a large
    learning rate would overwrite the language knowledge the reasoning task
    depends on - catastrophic forgetting. A low rate with a short warmup adapts
    the model while preserving what pretraining bought.

    Attributes:
        batch_size: Examples per optimizer step.
        learning_rate: Peak rate, an order of magnitude below pretraining's 6e-4.
        min_learning_rate: Floor of the cosine decay.
        warmup_steps: Linear warmup length.
        epochs: Passes over the training split.
        weight_decay: Applied to 2D weights only, as in pretraining.
        beta1: AdamW first-moment coefficient.
        beta2: AdamW second-moment coefficient.
        grad_clip: Gradient-norm clipping threshold.
        eval_interval: Steps between validation evaluations.
        log_interval: Steps between progress lines.
        seed: Seeds initialization and batch order.
        amp: Use mixed precision when CUDA is available.
        max_length: Longest tokenized example; longer ones are skipped.
    """

    batch_size: int = 32
    learning_rate: float = 5e-5
    min_learning_rate: float = 5e-6
    warmup_steps: int = 100
    epochs: int = 3
    weight_decay: float = 0.1
    beta1: float = 0.9
    beta2: float = 0.95
    grad_clip: float = 1.0
    eval_interval: int = 200
    log_interval: int = 50
    seed: int = 20260812
    amp: bool = True
    max_length: int = 64

    def to_dict(self) -> dict:
        """Serialize for checkpointing and the run record."""
        return asdict(self)

    @classmethod
    def from_dict(cls, data: dict) -> "FinetuneConfig":
        """Rebuild from a checkpoint payload, ignoring unknown keys."""
        fields = set(cls.__dataclass_fields__)
        return cls(**{k: v for k, v in data.items() if k in fields})


class ReasoningDataset:
    """Tokenized reasoning examples with answer-span label masking.

    Each example becomes ``[BOS] prompt_tokens answer_tokens [EOS]``, framed the
    same way Phase 2 framed corpus documents so the model sees a familiar shape.
    Labels are the input shifted by one, with every prompt position replaced by
    :data:`IGNORE_INDEX`.

    Args:
        examples: Rows loaded from the generated JSONL, each carrying ``prompt``
            and ``answer``.
        tokenizer: SentencePiece processor for this language.
        max_length: Examples longer than this are dropped rather than truncated -
            truncating would cut the answer off and train the model on a
            mislabelled target.
    """

    def __init__(self, examples: list[dict], tokenizer, max_length: int = 64) -> None:
        """Tokenize every example and build its masked label sequence."""
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.rows: list[dict] = []
        self.skipped = 0

        for row in examples:
            prompt_ids = tokenizer.encode(row["prompt"], out_type=int)
            full_ids = tokenizer.encode(f"{row['prompt']} {row['answer']}", out_type=int)

            # The prompt must be a token-level prefix of the full sequence, or the
            # answer span cannot be located by slicing. SentencePiece guarantees
            # this for the templates in use (verified), but a future template
            # could break it silently, so it is checked rather than assumed.
            if full_ids[: len(prompt_ids)] != prompt_ids:
                self.skipped += 1
                continue

            input_ids = [BOS_ID, *full_ids, EOS_ID]
            if len(input_ids) > max_length:
                self.skipped += 1
                continue

            # Labels predict position t+1 from position t. The BOS shifts
            # everything by one, so prompt positions occupy indices
            # 0 .. len(prompt_ids), and the answer begins right after.
            labels = list(input_ids)
            answer_start = len(prompt_ids) + 1  # +1 for BOS
            for index in range(min(answer_start, len(labels))):
                labels[index] = IGNORE_INDEX

            self.rows.append(
                {
                    "input_ids": input_ids,
                    "labels": labels,
                    "prompt": row["prompt"],
                    "answer": row["answer"],
                    "answer_tokens": len(full_ids) - len(prompt_ids),
                }
            )

    def __len__(self) -> int:
        """Number of usable examples."""
        return len(self.rows)

    def batches(self, batch_size: int, shuffle: bool, seed: int, device: str):
        """Yield padded ``(x, y)`` batches.

        Sequences are padded to the longest in the batch with :data:`PAD_ID`, and
        the corresponding label positions are set to :data:`IGNORE_INDEX` so
        padding contributes nothing to the loss.

        Args:
            batch_size: Examples per batch.
            shuffle: Shuffle example order; on for training, off for evaluation.
            seed: Seeds the shuffle, so batch order is reproducible.
            device: Where to place the tensors.

        Yields:
            ``(x, y)`` int64 tensors of shape ``(batch, length)``.
        """
        order = list(range(len(self.rows)))
        if shuffle:
            import random

            random.Random(seed).shuffle(order)

        for start in range(0, len(order), batch_size):
            chunk = [self.rows[i] for i in order[start : start + batch_size]]
            width = max(len(row["input_ids"]) for row in chunk)

            inputs, labels = [], []
            for row in chunk:
                pad = width - len(row["input_ids"])
                inputs.append(row["input_ids"] + [PAD_ID] * pad)
                labels.append(row["labels"] + [IGNORE_INDEX] * pad)

            batch_inputs = torch.tensor(inputs, dtype=torch.long)
            batch_labels = torch.tensor(labels, dtype=torch.long)

            # Inputs are positions 0..n-1, targets are positions 1..n, the same
            # shift pretraining used.
            x = batch_inputs[:, :-1].contiguous()
            y = batch_labels[:, 1:].contiguous()

            if device != "cpu":
                x = x.to(device, non_blocking=True)
                y = y.to(device, non_blocking=True)
            yield x, y

    def statistics(self) -> dict:
        """Summary of what was tokenized, for the run record."""
        lengths = [len(row["input_ids"]) for row in self.rows]
        answer_lengths = [row["answer_tokens"] for row in self.rows]
        return {
            "examples": len(self.rows),
            "skipped": self.skipped,
            "max_length": self.max_length,
            "sequence_tokens": {
                "min": min(lengths) if lengths else 0,
                "max": max(lengths) if lengths else 0,
                "mean": sum(lengths) / len(lengths) if lengths else 0.0,
            },
            "answer_tokens": {
                "min": min(answer_lengths) if answer_lengths else 0,
                "max": max(answer_lengths) if answer_lengths else 0,
                "single_token_fraction": (
                    sum(1 for n in answer_lengths if n == 1) / len(answer_lengths)
                    if answer_lengths
                    else 0.0
                ),
            },
        }


def finetune_lr(step: int, total_steps: int, cfg: FinetuneConfig) -> float:
    """Learning rate at `step`: linear warmup, then cosine decay.

    Same shape as pretraining's schedule, but over the finetuning horizon. As
    there, the schedule is a pure function of the step, so storing the step in a
    checkpoint stores the scheduler position.
    """
    if step < cfg.warmup_steps:
        return cfg.learning_rate * (step + 1) / max(cfg.warmup_steps, 1)

    progress = (step - cfg.warmup_steps) / max(total_steps - cfg.warmup_steps, 1)
    progress = min(max(progress, 0.0), 1.0)
    coefficient = 0.5 * (1.0 + math.cos(math.pi * progress))
    return cfg.min_learning_rate + coefficient * (
        cfg.learning_rate - cfg.min_learning_rate
    )


@dataclass
class FinetuneState:
    """Everything needed to resume a finetuning run mid-flight."""

    step: int = 0
    epoch: int = 0
    best_val_loss: float = float("inf")
    history: list[dict] = field(default_factory=list)
    wall_seconds: float = 0.0


class ReasoningFinetuner:
    """Finetunes one pretrained model on its own language's reasoning data.

    Args:
        model: A model already loaded with pretrained weights.
        train_data: Tokenized training examples.
        val_data: Tokenized validation examples.
        cfg: Optimization hyperparameters.
        out_dir: Where checkpoints and the history file are written.
        device: ``"cuda"`` or ``"cpu"``.
    """

    def __init__(
        self,
        model: TransformerLM,
        train_data: ReasoningDataset,
        val_data: ReasoningDataset,
        cfg: FinetuneConfig,
        out_dir: str | Path,
        device: str | None = None,
    ) -> None:
        """Move the model to the device and prepare the optimizer."""
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = model.to(self.device)
        self.train_data = train_data
        self.val_data = val_data
        self.cfg = cfg
        self.out_dir = Path(out_dir)
        self.out_dir.mkdir(parents=True, exist_ok=True)

        self.optimizer = build_optimizer(self.model, cfg)
        self.state = FinetuneState()

        self.amp_enabled = cfg.amp and self.device == "cuda"
        self.amp_dtype = torch.float32
        if self.amp_enabled:
            self.amp_dtype = (
                torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
            )
        self.scaler = torch.amp.GradScaler(
            "cuda", enabled=self.amp_enabled and self.amp_dtype is torch.float16
        )

        self.steps_per_epoch = max(
            1, math.ceil(len(train_data) / max(cfg.batch_size, 1))
        )
        self.total_steps = self.steps_per_epoch * cfg.epochs

        torch.manual_seed(cfg.seed)

    @torch.no_grad()
    def evaluate(self) -> dict:
        """Mean answer-span cross-entropy over the validation split.

        Because prompt positions are masked out, this loss is the model's
        uncertainty about the *answer* alone, which is the quantity the task
        cares about - unlike a whole-sequence loss, which would be dominated by
        the highly predictable template text.
        """
        self.model.eval()
        total_loss = 0.0
        total_batches = 0

        for x, y in self.val_data.batches(
            self.cfg.batch_size, shuffle=False, seed=self.cfg.seed, device=self.device
        ):
            with torch.autocast(
                device_type=self.device, dtype=self.amp_dtype, enabled=self.amp_enabled
            ):
                loss = self.model(x, targets=y)["loss"]
            total_loss += loss.float().item()
            total_batches += 1

        self.model.train()
        mean_loss = total_loss / max(total_batches, 1)
        return {
            "val_loss": mean_loss,
            "val_perplexity": math.exp(min(mean_loss, 20)),
            "batches": total_batches,
        }

    def save_checkpoint(self, name: str = "latest") -> Path:
        """Write a resume-capable checkpoint, atomically.

        Carries the same keys as a pretraining checkpoint - weights, optimizer
        state, step, and both configs - plus the pretrained checkpoint this run
        started from, so a finetuned model's provenance is recorded in the file
        rather than only in a notebook.
        """
        model = getattr(self.model, "_orig_mod", self.model)
        payload = {
            "model": model.state_dict(),
            "optimizer": self.optimizer.state_dict(),
            "scaler": self.scaler.state_dict(),
            "step": self.state.step,
            "epoch": self.state.epoch,
            "best_val_loss": self.state.best_val_loss,
            "history": self.state.history,
            "wall_seconds": self.state.wall_seconds,
            "model_config": model.config.to_dict(),
            "finetune_config": self.cfg.to_dict(),
            "torch_rng": torch.get_rng_state(),
            "phase": "finetune",
        }
        path = self.out_dir / f"finetuned_{name}.pt"
        tmp = path.with_suffix(".pt.tmp")
        torch.save(payload, tmp)
        tmp.replace(path)
        return path

    def load_checkpoint(self, path: str | Path) -> None:
        """Resume a finetuning run from one of its own checkpoints."""
        payload = torch.load(path, map_location=self.device, weights_only=False)
        model = getattr(self.model, "_orig_mod", self.model)
        model.load_state_dict(payload["model"])
        self.optimizer.load_state_dict(payload["optimizer"])
        if payload.get("scaler"):
            self.scaler.load_state_dict(payload["scaler"])

        self.state = FinetuneState(
            step=payload.get("step", 0),
            epoch=payload.get("epoch", 0),
            best_val_loss=payload.get("best_val_loss", float("inf")),
            history=payload.get("history", []),
            wall_seconds=payload.get("wall_seconds", 0.0),
        )

        if payload.get("torch_rng") is not None:
            try:
                state = payload["torch_rng"]
                if not isinstance(state, torch.Tensor):
                    state = torch.tensor(state, dtype=torch.uint8)
                torch.set_rng_state(state.cpu().to(torch.uint8))
            except (RuntimeError, TypeError) as error:
                print(f"  note: RNG state not restored ({error})")

    def train(self, resume: bool = True) -> FinetuneState:
        """Run the finetuning loop.

        Args:
            resume: Continue from ``finetuned_latest.pt`` when one exists.

        Returns:
            The final training state.
        """
        if resume:
            latest = self.out_dir / "finetuned_latest.pt"
            if latest.exists():
                self.load_checkpoint(latest)
                print(f"resumed at step {self.state.step:,}")

        self.model.train()
        started = time.time()

        while self.state.epoch < self.cfg.epochs:
            epoch_seed = self.cfg.seed + self.state.epoch
            for x, y in self.train_data.batches(
                self.cfg.batch_size, shuffle=True, seed=epoch_seed, device=self.device
            ):
                lr = finetune_lr(self.state.step, self.total_steps, self.cfg)
                for group in self.optimizer.param_groups:
                    group["lr"] = lr

                self.optimizer.zero_grad(set_to_none=True)
                with torch.autocast(
                    device_type=self.device,
                    dtype=self.amp_dtype,
                    enabled=self.amp_enabled,
                ):
                    loss = self.model(x, targets=y)["loss"]

                self.scaler.scale(loss).backward()
                if self.cfg.grad_clip > 0:
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(
                        self.model.parameters(), self.cfg.grad_clip
                    )
                self.scaler.step(self.optimizer)
                self.scaler.update()

                self.state.step += 1

                if self.state.step % self.cfg.log_interval == 0:
                    print(
                        f"step {self.state.step:>5,}/{self.total_steps:,}  "
                        f"loss {loss.float().item():.4f}  lr {lr:.2e}",
                        flush=True,
                    )

                if self.state.step % self.cfg.eval_interval == 0:
                    metrics = self.evaluate()
                    metrics.update(step=self.state.step, epoch=self.state.epoch, lr=lr)
                    self.state.history.append(metrics)
                    print(
                        f"  eval step {self.state.step:,}: "
                        f"val_loss {metrics['val_loss']:.4f}",
                        flush=True,
                    )
                    if metrics["val_loss"] < self.state.best_val_loss:
                        self.state.best_val_loss = metrics["val_loss"]
                        self.save_checkpoint("best")

            self.state.epoch += 1
            self.save_checkpoint("latest")

        final = self.evaluate()
        final.update(step=self.state.step, epoch=self.state.epoch, final=True)
        self.state.history.append(final)
        if final["val_loss"] < self.state.best_val_loss:
            self.state.best_val_loss = final["val_loss"]
            self.save_checkpoint("best")

        self.state.wall_seconds += time.time() - started
        self.save_checkpoint("latest")
        self._write_history()
        print(
            f"\nfinetuning complete: {self.state.step:,} steps, "
            f"best val_loss {self.state.best_val_loss:.4f}"
        )
        return self.state

    def _write_history(self) -> None:
        """Persist the metric history for the report's finetuning curve."""
        path = self.out_dir / "finetune_history.json"
        path.write_text(
            json.dumps(
                {
                    "step": self.state.step,
                    "epochs": self.state.epoch,
                    "best_val_loss": self.state.best_val_loss,
                    "wall_seconds": round(self.state.wall_seconds, 1),
                    "finetune_config": self.cfg.to_dict(),
                    "train_examples": len(self.train_data),
                    "val_examples": len(self.val_data),
                    "history": self.state.history,
                },
                indent=1,
            )
        )


def load_pretrained(checkpoint_path: str | Path, device: str) -> tuple[TransformerLM, dict]:
    """Rebuild a pretrained model from its Phase 2 checkpoint.

    The model architecture comes from the checkpoint's own stored config, so a
    finetuning run cannot silently disagree with the model it is loading.
    """
    payload = torch.load(checkpoint_path, map_location=device, weights_only=False)
    config = ModelConfig.from_dict(payload["model_config"])
    model = TransformerLM(config)
    model.load_state_dict(payload["model"])
    model.to(device)
    return model, payload

In [ ]:
%%writefile /kaggle/working/src/lma/reasoning/evaluate.py
"""Scoring a model on the reasoning task, before and after finetuning.

The headline number the assignment asks for is "pretrained vs. finetuned accuracy
(or exact match) on the synthetic test set". That comparison is only meaningful
if both models are scored the *same* way, so everything here runs identically
against either checkpoint - the pretrained model is not given an easier decoding
path or a more forgiving match rule.

How an answer is scored
-----------------------
Every answer in the generated corpus is a single token by construction (the
entity pools in :mod:`lma.reasoning.templates` are vetted against each language's
own tokenizer), so the model's answer is the token it puts highest probability on
immediately after the answer prefix. That makes exact match unambiguous and
removes decoding temperature as a confound: greedy argmax at one position, no
sampling, no length normalization.

Two accuracies are reported, and the gap between them is informative:

* **Free exact match** - argmax over the whole vocabulary. This is the honest
  measure of whether the model solved the task, and it is what the headline
  number reports.
* **Constrained accuracy** - argmax restricted to the entities named in that
  example's own premises. A pretrained model that has never seen the task will
  score near zero free but near chance constrained, which separates "cannot
  produce a name at all" from "produces a name but the wrong one".

Reporting only the first would understate what a pretrained model knows;
reporting only the second would overstate it.
"""

from __future__ import annotations

import math
from collections import defaultdict

import torch

from ..tokenizer.train_tokenizer import BOS_ID

#: Chance accuracy is 1/len(entities), which differs by chain length, so the
#: report breaks accuracy down per chain length rather than quoting one baseline.
#: See :func:`above_chance`, which is the comparison that should be quoted.


@torch.no_grad()
def score_examples(
    model,
    examples: list[dict],
    tokenizer,
    device: str | None = None,
    batch_size: int = 32,
) -> dict:
    """Evaluate one model on reasoning examples.

    Args:
        model: A :class:`~lma.model.TransformerLM`, pretrained or finetuned.
        examples: Rows carrying ``prompt``, ``answer``, ``entities``, and
            ``chain_length``.
        tokenizer: SentencePiece processor for this language.
        device: Where to run. Defaults to the model's device.
        batch_size: Prompts per forward pass.

    Returns:
        Overall and per-slice accuracy, mean answer log-probability, and the
        predictions themselves for qualitative inspection.
    """
    device = device or str(next(model.parameters()).device)
    model.eval()

    correct_free = 0
    correct_constrained = 0
    total = 0
    answer_logprob_sum = 0.0

    by_chain: dict[int, list[int]] = defaultdict(list)
    by_hops: dict[int, list[int]] = defaultdict(list)
    by_dimension: dict[str, list[int]] = defaultdict(list)
    by_polarity: dict[str, list[int]] = defaultdict(list)
    predictions: list[dict] = []

    for start in range(0, len(examples), batch_size):
        chunk = examples[start : start + batch_size]

        # Left-padding would shift the answer position per row; instead each
        # prompt is run to its own final position, so the logits read are always
        # the ones immediately following that prompt's last token.
        encoded = [[BOS_ID, *tokenizer.encode(row["prompt"], out_type=int)] for row in chunk]
        width = max(len(ids) for ids in encoded)
        padded = [ids + [0] * (width - len(ids)) for ids in encoded]
        batch = torch.tensor(padded, dtype=torch.long, device=device)

        logits = model(batch)["logits"]

        for index, row in enumerate(chunk):
            position = len(encoded[index]) - 1
            row_logits = logits[index, position, :].float()
            log_probs = torch.log_softmax(row_logits, dim=-1)

            answer_ids = tokenizer.encode(row["answer"], out_type=int)
            answer_id = answer_ids[0]

            predicted_id = int(row_logits.argmax().item())
            free_hit = int(predicted_id == answer_id)

            # Constrained: restrict to the entities this example actually names.
            candidate_ids = []
            for entity in row["entities"]:
                ids = tokenizer.encode(entity, out_type=int)
                candidate_ids.append(ids[0])
            candidate_tensor = torch.tensor(candidate_ids, device=row_logits.device)
            best_candidate = candidate_ids[
                int(row_logits[candidate_tensor].argmax().item())
            ]
            constrained_hit = int(best_candidate == answer_id)

            correct_free += free_hit
            correct_constrained += constrained_hit
            total += 1
            answer_logprob_sum += float(log_probs[answer_id].item())

            by_chain[row["chain_length"]].append(free_hit)
            by_hops[row.get("hops", row["chain_length"] - 1)].append(free_hit)
            by_dimension[row.get("dimension", "unknown")].append(free_hit)
            by_polarity["most" if row.get("ask_most") else "least"].append(free_hit)

            if len(predictions) < 40:
                predictions.append(
                    {
                        "prompt": row["prompt"],
                        "expected": row["answer"],
                        "predicted": tokenizer.id_to_piece(predicted_id).lstrip("▁"),
                        "correct": bool(free_hit),
                        "chain_length": row["chain_length"],
                    }
                )

    def rates(groups: dict) -> dict:
        """Convert per-slice hit lists into accuracy and support."""
        return {
            str(key): {
                "accuracy": sum(hits) / len(hits) if hits else 0.0,
                "n": len(hits),
            }
            for key, hits in sorted(groups.items(), key=lambda item: str(item[0]))
        }

    mean_logprob = answer_logprob_sum / max(total, 1)
    return {
        "examples": total,
        "exact_match": correct_free / max(total, 1),
        "constrained_accuracy": correct_constrained / max(total, 1),
        "answer_mean_logprob": mean_logprob,
        "answer_perplexity": math.exp(min(-mean_logprob, 20)),
        "by_chain_length": rates(by_chain),
        "by_hops": rates(by_hops),
        "by_dimension": rates(by_dimension),
        "by_polarity": rates(by_polarity),
        "predictions": predictions,
    }


def chance_baseline(examples: list[dict]) -> dict:
    """Accuracy a model would reach by guessing among each example's entities.

    Reported alongside the measured numbers because "above chance" is the only
    meaningful claim for a constrained-choice task, and chance differs by chain
    length: one in two for a two-entity comparison, one in four for a four-entity
    chain.
    """
    total = 0.0
    by_chain: dict[int, list[float]] = defaultdict(list)

    for row in examples:
        entities = len(row["entities"])
        probability = 1.0 / entities if entities else 0.0
        total += probability
        by_chain[row["chain_length"]].append(probability)

    return {
        "overall": total / max(len(examples), 1),
        "by_chain_length": {
            str(key): sum(values) / len(values)
            for key, values in sorted(by_chain.items())
        },
        "support_by_chain_length": {
            str(key): len(values) for key, values in sorted(by_chain.items())
        },
    }


def above_chance(results: dict, baseline: dict) -> dict:
    """Compare accuracy against chance *per chain length*, not in aggregate.

    Quoting one blended baseline is misleading here, because chance differs
    sharply by chain length - one in two for a two-entity comparison against one
    in four for a four-entity chain - so a corpus weighted toward short chains
    carries a high blended baseline that a model can beat on the hard examples
    and still miss overall. The per-length comparison is the honest one, and the
    aggregate is reported only alongside it.

    Args:
        results: :func:`score_examples` output.
        baseline: :func:`chance_baseline` output for the same examples.

    Returns:
        Per-length accuracy, chance, and margin, plus how many lengths clear it.
    """
    per_length = {}
    for key, payload in results.get("by_chain_length", {}).items():
        chance = baseline.get("by_chain_length", {}).get(key, 0.0)
        per_length[key] = {
            "accuracy": payload["accuracy"],
            "chance": chance,
            "margin": payload["accuracy"] - chance,
            "above_chance": payload["accuracy"] > chance,
            "n": payload["n"],
        }

    clearing = [key for key, row in per_length.items() if row["above_chance"]]
    return {
        "by_chain_length": per_length,
        "lengths_above_chance": sorted(clearing),
        "lengths_evaluated": sorted(per_length),
        "aggregate_accuracy": results.get("exact_match", 0.0),
        "aggregate_chance": baseline.get("overall", 0.0),
    }


def compare_pretrained_finetuned(before: dict, after: dict) -> dict:
    """Summarize what finetuning changed.

    Args:
        before: :func:`score_examples` output for the pretrained checkpoint.
        after: The same for the finetuned checkpoint.

    Returns:
        Absolute and relative deltas on the headline metrics, plus per-slice
        exact-match deltas.
    """
    delta_by_chain = {}
    for key, payload in after.get("by_chain_length", {}).items():
        prior = before.get("by_chain_length", {}).get(key, {}).get("accuracy", 0.0)
        delta_by_chain[key] = {
            "pretrained": prior,
            "finetuned": payload["accuracy"],
            "delta": payload["accuracy"] - prior,
            "n": payload["n"],
        }

    return {
        "exact_match": {
            "pretrained": before["exact_match"],
            "finetuned": after["exact_match"],
            "delta": after["exact_match"] - before["exact_match"],
        },
        "constrained_accuracy": {
            "pretrained": before["constrained_accuracy"],
            "finetuned": after["constrained_accuracy"],
            "delta": after["constrained_accuracy"] - before["constrained_accuracy"],
        },
        "answer_mean_logprob": {
            "pretrained": before["answer_mean_logprob"],
            "finetuned": after["answer_mean_logprob"],
            "delta": after["answer_mean_logprob"] - before["answer_mean_logprob"],
        },
        "by_chain_length": delta_by_chain,
    }


def error_analysis(results: dict, examples: list[dict]) -> dict:
    """Characterize the mistakes, for the report's qualitative section.

    Three failure modes are distinguished, because they call for different
    explanations:

    * **off_chain** - the predicted token is not any entity in the premises, so
      the model did not even identify the answer as a choice among the named
      entities.
    * **wrong_entity** - a named entity, but the wrong one: the model understood
      the format and failed the inference.
    * **inverted** - specifically the entity at the opposite end of the ordering,
      which suggests the model tracked the comparison but mishandled the
      question's polarity.
    """
    counts = {"off_chain": 0, "wrong_entity": 0, "inverted": 0}
    index = {row["prompt"]: row for row in examples}

    for prediction in results.get("predictions", []):
        if prediction["correct"]:
            continue
        row = index.get(prediction["prompt"])
        if row is None:
            continue
        predicted = prediction["predicted"]
        entities = row["entities"]

        if predicted not in entities:
            counts["off_chain"] += 1
            continue

        opposite = entities[-1] if row.get("ask_most") else entities[0]
        if predicted == opposite:
            counts["inverted"] += 1
        else:
            counts["wrong_entity"] += 1

    total = sum(counts.values())
    return {
        "counts": counts,
        "fractions": {
            key: (value / total if total else 0.0) for key, value in counts.items()
        },
        "errors_examined": total,
    }

In [ ]:
%%writefile /kaggle/working/src/lma/data/splits.py
"""Deterministic train/validation/test splitting.

Design decisions
----------------
**Hash-based assignment, not shuffling.** Each document's split is a function of
a hash of its content and a seed. This makes the split reproducible without
holding the corpus in memory, and stable under corpus growth: adding documents
never moves an existing document from train to test, which is what keeps the
Phase 2/3 evaluation numbers comparable across corpus revisions.

**Source-aware assignment.** The hash is salted per source so a source that
happens to dominate the corpus cannot skew one split. Optionally, whole sources
can be pinned to a split.

**Leakage control.** Documents are assigned by content hash, so exact duplicates
that survive cleaning land in the *same* split rather than straddling train and
test.
"""

from __future__ import annotations

import hashlib
import json
from dataclasses import dataclass, field
from pathlib import Path
from typing import Iterable, Iterator, TextIO

SPLIT_NAMES = ("train", "validation", "test")


@dataclass
class SplitConfig:
    """Split ratios and assignment policy.

    Attributes:
        train: Fraction assigned to training.
        validation: Fraction assigned to validation.
        test: Fraction assigned to test.
        seed: Hash salt; changing it produces a different but equally
            reproducible partition.
        pinned_sources: Optional ``{source_id: split}`` overrides, for holding a
            whole source out (e.g. keeping one OCR'd book entirely in test).
    """

    train: float = 0.98
    validation: float = 0.01
    test: float = 0.01
    seed: int = 20260812
    pinned_sources: dict[str, str] = field(default_factory=dict)

    def __post_init__(self) -> None:
        """Validate that ratios are positive and sum to 1."""
        total = self.train + self.validation + self.test
        if abs(total - 1.0) > 1e-6:
            raise ValueError(f"Split ratios must sum to 1.0, got {total}")
        if min(self.train, self.validation, self.test) < 0:
            raise ValueError("Split ratios must be non-negative")
        for source, split in self.pinned_sources.items():
            if split not in SPLIT_NAMES:
                raise ValueError(f"Pinned source {source!r} has invalid split {split!r}")


class SplitAssigner:
    """Assigns documents to splits by salted content hash."""

    def __init__(self, config: SplitConfig) -> None:
        """Precompute the cumulative split boundaries."""
        self.config = config
        self._train_edge = config.train
        self._val_edge = config.train + config.validation

    def assign(self, text: str, source_id: str = "") -> str:
        """Return the split name for one document.

        Args:
            text: Document text; only its hash is used.
            source_id: Source identifier, used for salting and for pinning.

        Returns:
            One of ``"train"``, ``"validation"``, ``"test"``.
        """
        pinned = self.config.pinned_sources.get(source_id)
        if pinned is not None:
            return pinned

        payload = f"{self.config.seed}:{source_id}:{' '.join(text.split())}".encode("utf-8")
        digest = hashlib.blake2b(payload, digest_size=8).digest()
        position = int.from_bytes(digest, "big") / float(1 << 64)

        if position < self._train_edge:
            return "train"
        if position < self._val_edge:
            return "validation"
        return "test"


class SplitWriter:
    """Writes documents into one JSONL file per split, tracking counts.

    Intended for use as a context manager so file handles always close::

        with SplitWriter(Path("data/processed"), "bn") as writer:
            writer.write("train", text, "wikipedia", "downloaded")
    """

    def __init__(self, output_dir: Path, language: str, append: bool = False) -> None:
        """Prepare the output directory and per-split counters.

        Args:
            output_dir: Directory to write the per-split JSONL files into.
            language: ISO code, used in the output filenames.
            append: Open the split files for appending instead of truncating, and
                start the counters from `restore_counts`. Required when resuming a
                checkpointed run - opening with ``"w"`` would silently discard
                every document written before the interruption.
        """
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.language = language
        self.append = append
        self._handles: dict[str, TextIO] = {}
        self.counts: dict[str, int] = {name: 0 for name in SPLIT_NAMES}
        self.chars: dict[str, int] = {name: 0 for name in SPLIT_NAMES}
        self.by_collection: dict[str, dict[str, int]] = {name: {} for name in SPLIT_NAMES}
        self.by_source: dict[str, dict[str, int]] = {name: {} for name in SPLIT_NAMES}

    def restore_counts(self, summary: dict) -> None:
        """Seed the counters from a previous run's :meth:`summary` output.

        Called when resuming so the final statistics cover the whole corpus rather
        than only the documents written since the last checkpoint.
        """
        self.counts.update({k: v for k, v in summary.get("documents", {}).items() if k in self.counts})
        self.chars.update({k: v for k, v in summary.get("characters", {}).items() if k in self.chars})
        for split, payload in summary.get("characters_by_collection", {}).items():
            if split in self.by_collection:
                self.by_collection[split] = dict(payload)
        for split, payload in summary.get("characters_by_source", {}).items():
            if split in self.by_source:
                self.by_source[split] = dict(payload)

    def __enter__(self) -> "SplitWriter":
        """Open one output file per split, truncating unless appending."""
        mode = "a" if self.append else "w"
        for name in SPLIT_NAMES:
            path = self.output_dir / f"{self.language}_{name}.jsonl"
            self._handles[name] = path.open(mode, encoding="utf-8")
        return self

    def __exit__(self, *exc_info) -> None:
        """Flush and close all output files."""
        for handle in self._handles.values():
            handle.flush()
            handle.close()
        self._handles.clear()

    def write(self, split: str, text: str, source_id: str, collection: str) -> None:
        """Append one document to a split and update the counters."""
        handle = self._handles[split]
        handle.write(json.dumps({"text": text, "source_id": source_id, "collection": collection}, ensure_ascii=False))
        handle.write("\n")
        self.counts[split] += 1
        self.chars[split] += len(text)
        self.by_collection[split][collection] = self.by_collection[split].get(collection, 0) + len(text)
        self.by_source[split][source_id] = self.by_source[split].get(source_id, 0) + len(text)

    def flush(self) -> None:
        """Force buffered documents to disk, so a checkpoint matches the files.

        A checkpoint records how many documents have been written. If the OS still
        holds some in a buffer when the process dies, the files would be shorter
        than the checkpoint claims and the resumed run would double-count.
        """
        for handle in self._handles.values():
            handle.flush()

    def summary(self) -> dict:
        """Return per-split document counts, characters, and provenance breakdowns."""
        return {
            "language": self.language,
            "documents": dict(self.counts),
            "characters": dict(self.chars),
            "characters_by_collection": {k: dict(v) for k, v in self.by_collection.items()},
            "characters_by_source": {k: dict(v) for k, v in self.by_source.items()},
            "files": {name: str(self.output_dir / f"{self.language}_{name}.jsonl") for name in SPLIT_NAMES},
        }


def iter_jsonl(path: Path, text_field: str = "text") -> Iterator[dict]:
    """Yield parsed rows from a JSONL file, skipping malformed lines."""
    with Path(path).open("r", encoding="utf-8") as handle:
        for line in handle:
            line = line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except json.JSONDecodeError:
                continue
            if text_field in row:
                yield row


def iter_texts(paths: Iterable[Path], text_field: str = "text") -> Iterator[str]:
    """Yield just the text field from a sequence of JSONL files."""
    for path in paths:
        for row in iter_jsonl(Path(path), text_field):
            yield row[text_field]

## 3. Find the pretrained checkpoint and the tokenizer

Both are attached datasets, so this searches `/kaggle/input` recursively rather
than assuming a fixed depth - Kaggle nests a dataset under its own name, and the
producing notebook may have written into a subfolder.

In [ ]:
import torch
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
OUT = WORK / f"{LANGUAGE}_finetuned"
OUT.mkdir(parents=True, exist_ok=True)

# Prefer the best checkpoint (lowest validation loss) over the latest one, and
# only one belonging to THIS language - attaching both languages' datasets must
# not silently finetune the wrong model.
candidates = [
    p for p in list(INPUT.rglob("checkpoint_best.pt")) + list(INPUT.rglob("checkpoint_latest.pt"))
    if LANGUAGE in str(p).lower()
]
if not candidates:
    attached = sorted(p.name for p in INPUT.iterdir()) if INPUT.exists() else []
    raise FileNotFoundError(
        f"No checkpoint for {LANGUAGE} under /kaggle/input. Attached: {attached or 'none'}"
    )
CHECKPOINT = sorted(candidates)[0]
print("checkpoint:", CHECKPOINT)

tokenizers = sorted(INPUT.rglob(f"{LANGUAGE}_unigram_*.model"))
if tokenizers:
    TOKENIZER = tokenizers[0]
else:
    # The tokenizers are small and committed to the repository, so fall back to
    # a copy written by the cell below if no dataset supplied one.
    TOKENIZER = WORK / f"{LANGUAGE}_unigram_8000.model"
print("tokenizer:", TOKENIZER)

### If the tokenizer was not attached

The `.model` files live in the repository under `hindi/tokenizer/` and
`nepali/tokenizer/`. Upload the one for this language as a dataset, or paste it
here. The cell below only checks; it does not invent a tokenizer, because a
different vocabulary would silently invalidate the pretrained weights.

In [ ]:
if not Path(TOKENIZER).exists():
    raise FileNotFoundError(
        f"{TOKENIZER} not found. Attach the Phase 1 tokenizer for {LANGUAGE} as a "
        "dataset (hindi/tokenizer/ or nepali/tokenizer/ in the repository)."
    )

import sentencepiece as spm

tokenizer = spm.SentencePieceProcessor(model_file=str(TOKENIZER))
print("vocabulary:", tokenizer.get_piece_size())

## 4. Generate the reasoning dataset

Generated here from the committed generator rather than uploaded, so the notebook
is self-contained. Generation is seeded, so this reproduces the corpus in the
repository byte for byte - and the leakage checks below are the same ones
`scripts/generate_reasoning_data.py` enforces.

In [ ]:
from lma.reasoning.generate import (
    GeneratorConfig, ReasoningDatasetBuilder, dataset_statistics,
)

builder = ReasoningDatasetBuilder(
    GeneratorConfig(language=LANGUAGE, total_examples=TOTAL_EXAMPLES)
)
examples = builder.build()
stats = dataset_statistics(examples, builder)

print("split sizes:", stats["by_split"])
print("chain lengths:", stats["by_chain_length"])
print()
print("leakage checks:")
for name, value in stats["leakage_checks"].items():
    print(f"  {name}: {value if not isinstance(value, list) else len(value)}")

violations = [
    name for name in (
        "chain_signature_overlap", "prompt_overlap_train_eval",
        "duplicate_prompts", "distractor_answers_in_eval",
    )
    if stats["leakage_checks"][name]
]
assert not violations, f"leakage checks failed: {violations}"
print("\nall leakage checks passed")

In [ ]:
print("sample training examples:\n")
for example in examples[:4]:
    print(" ", example.text)
    print(f"    answer={example.answer}  chain={example.chain_length}  hops={example.hops}")

## 5. Load the pretrained model

The architecture comes from the checkpoint's own stored config, so a finetuning
run cannot silently disagree with the model it is loading.

In [ ]:
from lma.reasoning.finetune import (
    FinetuneConfig, ReasoningDataset, ReasoningFinetuner, load_pretrained,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model, payload = load_pretrained(CHECKPOINT, device)

print(f"parameters      : {model.num_parameters():,}")
print(f"pretrained step : {payload.get('step', 0):,}")
print(f"tokens seen     : {payload.get('tokens_seen', 0) / 1e6:.0f}M")
print(f"device          : {device}")

## 6. Score the pretrained model first

This is the "before" half of the comparison the assignment asks for, and it has
to be measured **before** finetuning touches the weights.

In [ ]:
from lma.reasoning.evaluate import (
    above_chance, chance_baseline, compare_pretrained_finetuned, error_analysis,
    score_examples,
)

test_rows = [e.to_dict() for e in examples if e.split == "test"]
train_rows = [e.to_dict() for e in examples if e.split == "train"]
val_rows = [e.to_dict() for e in examples if e.split == "validation"]

before = score_examples(model, test_rows, tokenizer, device=device)
baseline = chance_baseline(test_rows)

print(f"pretrained exact match : {before['exact_match']:.4f}")
print(f"constrained accuracy   : {before['constrained_accuracy']:.4f}")
print(f"chance (aggregate)     : {baseline['overall']:.4f}")

## 7. Finetune

Loss is computed on the **answer span only**: the prompt states premises the
model is conditioned on, not asked to produce. Training it to generate the
premises would spend capacity on text the task always supplies.

In [ ]:
config = FinetuneConfig(
    batch_size=BATCH_SIZE, epochs=EPOCHS, learning_rate=LEARNING_RATE,
)

train_data = ReasoningDataset(train_rows, tokenizer, config.max_length)
val_data = ReasoningDataset(val_rows, tokenizer, config.max_length)
print(f"train {len(train_data)}, validation {len(val_data)}")
print("tokenization:", train_data.statistics()["sequence_tokens"])

finetuner = ReasoningFinetuner(model, train_data, val_data, config, OUT, device=device)
state = finetuner.train(resume=True)

## 8. Score the finetuned model

Same examples, same decoding, same match rule - the only thing that changed is
the weights.

In [ ]:
after = score_examples(model, test_rows, tokenizer, device=device)
comparison = compare_pretrained_finetuned(before, after)
margins = above_chance(after, baseline)

print(f"exact match : {before['exact_match']:.4f} -> {after['exact_match']:.4f}"
      f"  (delta {comparison['exact_match']['delta']:+.4f})")
print(f"constrained : {before['constrained_accuracy']:.4f} -> {after['constrained_accuracy']:.4f}")
print()
print("per chain length (finetuned):")
for key, row in sorted(margins["by_chain_length"].items()):
    verdict = "above" if row["above_chance"] else "below"
    print(f"  {key} entities: acc {row['accuracy']:.3f}  chance {row['chance']:.3f}"
          f"  margin {row['margin']:+.3f}  n={row['n']}  ({verdict} chance)")

In [ ]:
print("error profile (finetuned):")
errors = error_analysis(after, test_rows)
for name, fraction in errors["fractions"].items():
    print(f"  {name:14s} {fraction:.3f}")

print("\nsample predictions:")
for prediction in after["predictions"][:8]:
    mark = "OK " if prediction["correct"] else "BAD"
    print(f"  [{mark}] expected={prediction['expected']:10s} "
          f"predicted={prediction['predicted']:10s} chain={prediction['chain_length']}")

## 9. Attention: pretrained against finetuned

The same comparative-reasoning prompt through both models, so any difference in
where the heads look is attributable to finetuning.

In [ ]:
from lma.eval import attention_statistics, classify_heads, plot_attention_heatmaps, plot_head_summary
from lma.tokenizer.train_tokenizer import BOS_ID

FIGURES = WORK / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

prompt = test_rows[0]["prompt"]
print("prompt:", prompt, "\n")

# The pretrained weights were overwritten in place by finetuning, so reload them
# for the comparison rather than trusting the in-memory model.
pretrained_model, _ = load_pretrained(CHECKPOINT, device)

attention_summary = {}
for name, this_model in (("pretrained", pretrained_model), ("finetuned", model)):
    ids = torch.tensor([[BOS_ID, *tokenizer.encode(prompt, out_type=int)]], dtype=torch.long)
    with torch.no_grad():
        attentions = this_model(ids.to(device), return_attention=True)["attentions"]

    stats_a = attention_statistics(attentions)
    groups = classify_heads(stats_a)
    attention_summary[name] = {
        "layer_mean_entropy": stats_a["layer_mean_entropy"],
        "layer_mean_distance": stats_a["layer_mean_distance"],
        "normalized_entropy": stats_a["normalized_entropy"],
        "mean_distance": stats_a["mean_distance"],
        "head_groups": groups["counts"],
    }
    print(f"{name:11s} head types: {groups['counts']}")

    pieces = [tokenizer.id_to_piece(int(t)) for t in ids[0].tolist()]
    heads = list(range(min(3, this_model.config.n_head)))
    plot_attention_heatmaps(
        attentions, pieces,
        layers=[0, this_model.config.n_layer - 1], heads=heads,
        out_path=FIGURES / f"reasoning_attention_{name}_{LANGUAGE}.png",
        title=f"{LANGUAGE} reasoning prompt - {name}",
    )
    plot_head_summary(
        stats_a, FIGURES / f"reasoning_head_summary_{name}_{LANGUAGE}.png",
        title=f"{LANGUAGE} per-head statistics - {name}",
    )

print()
print("layer  pretrained  finetuned   delta")
shifts = []
for layer, (pre, post) in enumerate(zip(
    attention_summary["pretrained"]["layer_mean_distance"],
    attention_summary["finetuned"]["layer_mean_distance"],
)):
    shifts.append({"layer": layer, "pretrained": pre, "finetuned": post, "delta": post - pre})
    print(f"{layer:5d}  {pre:10.2f}  {post:9.2f}  {post - pre:+6.2f}")
attention_summary["distance_shift_by_layer"] = shifts

In [ ]:
from IPython.display import Image, display

for name in ("pretrained", "finetuned"):
    print(name)
    display(Image(filename=str(FIGURES / f"reasoning_attention_{name}_{LANGUAGE}.png")))

## 10. Save the results

Everything the Phase 3 report needs. Download `phase3_evaluation.json` and place
it at `<language>/reports/phase3_evaluation.json` in the repository, copy the
figures into `report/figures/`, then run `scripts/make_phase3_report.py`.

In [ ]:
import json

results = {
    "language": LANGUAGE,
    "model": "Model H" if LANGUAGE == "hi" else "Model L",
    "test_examples": len(test_rows),
    "pretrained_checkpoint": str(CHECKPOINT),
    "finetuned_checkpoint": str(OUT / "finetuned_best.pt"),
    "pretrained_step": payload.get("step"),
    "parameters": model.num_parameters(),
    "model_config": model.config.to_dict(),
    "finetune_config": config.to_dict(),
    "chance_baseline": baseline,
    "pretrained": before,
    "finetuned": after,
    "comparison": comparison,
    "above_chance": margins,
    "error_analysis": {
        "pretrained": error_analysis(before, test_rows),
        "finetuned": errors,
    },
    "attention": attention_summary,
    "attention_prompt": prompt,
    "dataset": stats,
}

path = WORK / "phase3_evaluation.json"
path.write_text(json.dumps(results, indent=1, ensure_ascii=False))
print("wrote", path)

record = {
    "language": LANGUAGE,
    "pretrained_checkpoint": str(CHECKPOINT),
    "pretrained_step": payload.get("step"),
    "parameters": model.num_parameters(),
    "model_config": model.config.to_dict(),
    "finetune_config": config.to_dict(),
    "steps": state.step,
    "epochs": state.epoch,
    "best_val_loss": state.best_val_loss,
    "wall_seconds": round(state.wall_seconds, 1),
    "train_data": train_data.statistics(),
    "val_data": val_data.statistics(),
    "history": state.history,
}
record_path = WORK / "finetune_record.json"
record_path.write_text(json.dumps(record, indent=1, ensure_ascii=False))
print("wrote", record_path)

## 11. Save the checkpoint

**Do not skip this.** `/kaggle/working` is wiped when the session ends.

Save the notebook version, then from the **Output** tab create a dataset named
`lma-phase3-<language>-finetuned`. Put its Drive link in the README, which the
submission instructions require for the finetuned checkpoints.

In [ ]:
for path in sorted(OUT.iterdir()):
    print(f"{path.stat().st_size / 1e6:9.1f} MB  {path.name}")

print(f"\nSave as a Kaggle Dataset named lma-phase3-{LANGUAGE}-finetuned")
print("Then upload finetuned_best.pt to Google Drive and put the link in README.md")